In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import pandas as pd
from torch.utils.data import Dataset
import torch.nn.init as init
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import numpy as np
import os
import random
from PIL import Image
from torchvision import models, transforms, utils
from torchvision.transforms import v2
import re



import copy

# Verificar si CUDA está disponible
cuda_available = torch.cuda.is_available()

# Imprimir la versión de PyTorch y la disponibilidad de CUDA
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {cuda_available}")

# Imprimir el nombre de la GPU si CUDA está disponible
if cuda_available:
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    # Obtener la versión de CUDA
    cuda_version = torch.version.cuda
    print(f"CUDA version: {cuda_version}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

PyTorch version: 2.2.0
CUDA available: True
GPU name: NVIDIA GeForce RTX 3050 Laptop GPU
CUDA version: 12.1
Using device: cuda


In [2]:
####CREACION DATASET

import os
import pandas as pd

# Directorio que contiene los archivos
directorio = './IMGS_MOSAIC'

# Obtener la lista de archivos en el directorio
archivos = os.listdir(directorio)

# Crear una lista para almacenar las rutas de los archivos y las etiquetas
data = []

# Iterar sobre los archivos
for archivo in archivos:
    # Obtener la ruta completa del archivo
    ruta_completa = os.path.join(directorio, archivo)
    # Obtener los elementos después de hacer split("_")
    elementos = archivo.split("_")
    # Obtener la etiqueta (cuarto elemento)
    etiqueta = elementos[3] if len(elementos) >= 4 else None
    # Agregar la ruta y la etiqueta a la lista de datos
    data.append((ruta_completa, int(etiqueta)))

# Crear el DataFrame de Pandas
df_mosaic = pd.DataFrame(data, columns=['Ruta', 'Etiqueta'])

# Mostrar el DataFrame
print(df_mosaic)

                             Ruta  Etiqueta
0   ./IMGS_MOSAIC\_P02_HVN_1_.jpg         1
1   ./IMGS_MOSAIC\_P08_HVN_1_.jpg         1
2   ./IMGS_MOSAIC\_P10_HVN_1_.jpg         1
3   ./IMGS_MOSAIC\_P12_HVN_0_.jpg         0
4   ./IMGS_MOSAIC\_P15_HVN_0_.jpg         0
5   ./IMGS_MOSAIC\_P17_HVN_0_.jpg         0
6   ./IMGS_MOSAIC\_P19_HVN_0_.jpg         0
7   ./IMGS_MOSAIC\_P20_HVN_0_.jpg         0
8   ./IMGS_MOSAIC\_P21_HVN_1_.jpg         1
9   ./IMGS_MOSAIC\_P22_HVN_1_.jpg         1
10  ./IMGS_MOSAIC\_P24_HVN_1_.jpg         1
11  ./IMGS_MOSAIC\_P25_HVN_0_.jpg         0
12  ./IMGS_MOSAIC\_P26_HVN_0_.jpg         0
13  ./IMGS_MOSAIC\_P27_HVN_1_.jpg         1
14  ./IMGS_MOSAIC\_P34_HVN_0_.jpg         0
15  ./IMGS_MOSAIC\_P47_HVN_0_.jpg         0
16  ./IMGS_MOSAIC\_P48_HVN_0_.jpg         0
17  ./IMGS_MOSAIC\_P50_H9O_1_.jpg         1
18  ./IMGS_MOSAIC\_P51_H9O_0_.jpg         0
19  ./IMGS_MOSAIC\_P52_H9O_1_.jpg         1
20  ./IMGS_MOSAIC\_P53_H9O_0_.jpg         0
21  ./IMGS_MOSAIC\_P54_H9O_1_.jp

## FUNCIÓN SIN TL SIN OC

In [3]:

def accuracy(outputs, labels):
    preds = outputs > 0.5  # Si la salida es mayor que 0.5, consideramos que es positiva
    correct = (preds == labels.byte())  # Compara las predicciones con las etiquetas reales
    acc = correct.float().sum() / labels.size(0)  # Calcula la precisión
    return acc.item()

def recall(outputs, labels):
    preds = outputs > 0.5

    preds = preds.cpu()
    labels = labels.cpu()
    TP = sum((preds == 1) & (labels == 1))
    FN = sum((preds == 0) & (labels == 1))

    recall_score = TP / (TP + FN) if (TP + FN) > 0 else 0
    return recall_score

def precision(outputs, labels):
    preds = outputs > 0.5

    preds = preds.cpu()
    labels = labels.cpu()

    TP = sum((preds == 1) & (labels == 1))
    FP = sum((preds == 1) & (labels == 0))

    precision_score = TP / (TP + FP) if (TP + FP) > 0 else 0
    return precision_score



def prueba_k_fold_resnet_no_pre(dataset, num_folds_outer, num_epochs, op, batch_size, lr, data_aug, ini, random_seed, seeds_tr_split):
    # Transformaciones para preprocesar las imágenes
    if data_aug:
        transform_t = v2.Compose([
            v2.Resize((224, 224)),
            # v2.RandomResizedCrop(224),
            v2.RandomHorizontalFlip(),
            v2.RandomVerticalFlip(),
            # v2.ElasticTransform(alpha=250.0),
            # v2.RandomRotation(45),
            # transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.2),
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

        transform_v = v2.Compose([
            v2.Resize((224, 224)),
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])
    else:
        transform_t = v2.Compose([
            v2.Resize((224, 224)),
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

        transform_v = v2.Compose([
            v2.Resize((224, 224)),
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

    # Modelo personalizado basado en ResNet
    class CustomResNet(nn.Module):
        def __init__(self, num_classes_binary=1):
            super(CustomResNet, self).__init__()
            resnet = models.resnet18(weights='IMAGENET1K_V1')
            # Extraer las capas de la red ResNet, excluyendo la capa de clasificación
            self.features = nn.Sequential(*list(resnet.children())[:-1])


            # Personalizar la capa de salida para tus necesidades
            self.fc_binary = nn.Linear(512, num_classes_binary)

            if (ini== "G"):
                # Inicialización de Glorot para las capas fully connected
                self.apply(self.init_weights_glorot)
            elif(ini == "H"):
                self.apply(self.init_weights_he)
            elif (ini== "N"):
                self.apply(self.init_weights_normal)
            elif(ini == "U"):
                self.apply(self.init_weights_uniform)
            elif (ini== "O"):
                self.apply(self.init_weights_orthogonal)
            elif(ini == "Z"):
                self.apply(self.init_weights_zeros)
            elif(ini == "ON"):
                self.apply(self.init_weights_ones)

        def init_weights_glorot(self, m):
            if isinstance(m, nn.Linear):
                init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    init.constant_(m.bias, 0)

        def init_weights_he(self, m):
            if isinstance(m, nn.Linear):
                init.kaiming_uniform_(m.weight, mode='fan_in', nonlinearity='relu')
                if m.bias is not None:
                    init.constant_(m.bias, 0)
        def init_weights_normal(self, m):
            if isinstance(m, nn.Linear):
                init.normal_(m.weight, mean=0, std=0.01)  # Puedes ajustar la media y la desviación estándar según tus necesidades
                if m.bias is not None:
                    init.constant_(m.bias, 0)

        def init_weights_uniform(self, m):
            if isinstance(m, nn.Linear):
                init.uniform_(m.weight, a=-0.1, b=0.1)  # Puedes ajustar los límites inferior y superior según tus necesidades
                if m.bias is not None:
                    init.constant_(m.bias, 0)

        def init_weights_orthogonal(self, m):
            if isinstance(m, nn.Linear):
                init.orthogonal_(m.weight)  # Puedes ajustar el parámetro 'gain' según tus necesidades
                if m.bias is not None:
                    init.constant_(m.bias, 0)

        def init_weights_zeros(self, m):
            if isinstance(m, nn.Linear):
                init.zeros_(m.weight)
                if m.bias is not None:
                    init.zeros_(m.bias)

        def init_weights_ones(self, m):
            if isinstance(m, nn.Linear):
                init.ones_(m.weight)
                if m.bias is not None:
                    init.ones_(m.bias)

        def forward(self, x):
            x = self.features(x)
            x = x.view(x.size(0), -1)


            # Salida para la variable binaria
            out_binary = self.fc_binary(x)

            return out_binary

    # Definir una clase para el conjunto de datos personalizado
    ##adaptar esto para meter unas rutsa y obtener dataos y etiquetas
    class CustomDataset(Dataset):
        def __init__(self, dataframe, transform=None):
            self.data = dataframe
            self.transform = transform

        def __len__(self):
            return len(self.data)

        def __getitem__(self, idx):
            img_path = self.data.iloc[idx, 0]  # Ruta de la imagen en la primera columna
            image = Image.open(img_path).convert('RGB')
            # Etiquetas: una variable binaria
            label_binary = torch.tensor(self.data.iloc[idx, 1], dtype=torch.float32)

            if self.transform:
                image = self.transform(image)

            return image, label_binary

    #####5 fold CV
    from sklearn.model_selection import StratifiedKFold

    # Definir el número de folds
    criterion_binary = nn.BCEWithLogitsLoss()
    kf_outer = StratifiedKFold(n_splits=num_folds_outer, shuffle=True, random_state=random_seed)
    ##No será necesario el inner porque la partición será fija en 80%/20%
    # kf_inner = StratifiedKFold(n_splits=num_folds_inner, shuffle=True, random_state=14)
    # Listas para almacenar las pérdidas de entrenamiento y validación de cada fold
    accs = []
    recs = []
    precs = []
    

    
    # Aqui se itera sobre los folds outer, es decir, uno de los folds se usará para test, ya que el modelo no lo verá durante el entrenamiento.

    ##Por otro lado, la parte de train, se verá otra vez sometida a un kfold, 

    for fold, (train_index, test_index) in enumerate(kf_outer.split(np.zeros(len(dataset)),dataset['Etiqueta'])):
        print(f"Fold {fold + 1}")

        # Dividir el conjunto de datos en entrenamiento y validación para este fold
        train_data_fold = dataset.iloc[train_index]
        test_data_fold = dataset.iloc[test_index]

        ##Aqui ya tengo train y test por lo que, antes del bucle para las épocas, se hará la partición con sklearn train test split para validación y train
        ##como se hace dentro del bucle para el outer CV, se hará una vez por fold, que es justo lo que queremos para promediar su rendimiento en un conjunto de test
        ##el cual no se ha usado en ningún momento para entrenar
        train_data_inner, val_data_inner = train_test_split(train_data_fold, test_size=0.2, random_state=seeds_tr_split[fold-1], stratify = train_data_fold['Etiqueta'])
        # Crear instancias del conjunto de datos personalizado para este fold
        train_dataset_fold = CustomDataset(dataframe=train_data_inner, transform=transform_t)
        val_dataset_fold = CustomDataset(dataframe=val_data_inner, transform=transform_v)

        test_dataset = CustomDataset(dataframe=test_data_fold, transform=transform_v)
        # Crear DataLoader para cargar los datos en lotes
        train_loader_fold_in = DataLoader(dataset=train_dataset_fold, batch_size=batch_size, shuffle=True)
        val_loader_fold_in = DataLoader(dataset=val_dataset_fold, batch_size=batch_size, shuffle=False)

        ##Aqui el batch size será de la longitud del test_dataset
        ##como se usa con model eval daría un poco igual el batch size, como en validacion
        test_dataloader = DataLoader(dataset=test_dataset, batch_size=len(test_data_fold), shuffle=False)
       
        # Re-inicializar el modelo para cada fold
        model = CustomResNet(num_classes_binary=1).to(device)
        optimizer = op(model.parameters(), lr = lr)
        
        # Entrenamiento y validación para este fold
        
        val_loss_best = -1
        best_epoc = -1
        best_model = None
        for epoch in range(num_epochs):
            # Entrenamiento del modelo
            # Fase de entrenamiento
            model.train()
            loss_total_acumulada = 0.0
            
            for images, bin_label in train_loader_fold_in:
                # for i in images:
                #     print(i.shape)
                # images_grid = utils.make_grid(images, nrow=8)  # Hacer una cuadrícula de imágenes
                # plt.figure(figsize=(10, 10))
                # plt.imshow(np.transpose(images_grid, (1, 2, 0)))  # Transponer las dimensiones para mostrar correctamente
                # plt.axis('off')
                # plt.show()
                # break  # Salir después de mostrar el primer lote
                # Reiniciar gradientes
                # Reiniciar gradientes
                images, bin_label = images.to(device), bin_label.to(device)
                optimizer.zero_grad()
                # Forward pass
                bin_out = model(images)
                # Calcular pérdida para las variables continuas
                # Calcular pérdida para la variable binaria
                lossT_bin = criterion_binary(bin_out.squeeze(1), bin_label)
                # Pérdida total
                loss_total = lossT_bin
                loss_total_acumulada += loss_total
                # Backward pass y optimización
                loss_total.backward()
                optimizer.step()
                
            # Validación del modelo
            model.eval()
            val_loss = 0.0
            
            with torch.no_grad():
                for images, bin_label in val_loader_fold_in:
                    # Reiniciar gradientes
                    images,  bin_label = images.to(device),  bin_label.to(device)
                    ##print(len(images)) aqui se hace con 32 imagenes
                    # Forward pass en el conjunto de validación
                    bin_out = model(images)
                    # Calcular pérdida para la variable binaria
                    lossV_bin = criterion_binary(bin_out.squeeze(1), bin_label)
                    # Sumar las pérdidas del conjunto de validación
                    val_loss += lossV_bin.item()
                    

            avg_loss_total_acumulada = loss_total_acumulada / len(train_loader_fold_in)
            
            avg_val_loss = val_loss / len(val_loader_fold_in)
            if(val_loss_best == -1):
                val_loss_best = avg_val_loss
                best_model = copy.deepcopy(model)
                best_epoc = epoch
            else:
                if (avg_val_loss < val_loss_best):
                    val_loss_best = avg_val_loss
                    best_model = copy.deepcopy(model)
                    best_epoc = epoch

            
        ####Aqui es justo antes del cambio de fold, ya tenemos nuestro mejor modelo en base al loss y además tenemos su valor, lo sacamos por pantalla
        print(f"Best epoc: {best_epoc}. Best val loss: {val_loss_best}")

        ##Ahora llegaría el turno de hacer la predicción en test, tanto precision, como recall como accuracy
        acc = 0
        rec = 0
        prec = 0
        best_model.eval()

        with torch.no_grad():
            for images, bin_label in test_dataloader:
                   
                    images, bin_label = images.to(device), bin_label.to(device)
                    
                    bin_out = best_model(images)
                    bin_pred = torch.sigmoid(bin_out.squeeze())
    
                    acc += accuracy(bin_pred, bin_label)
                    rec += recall(bin_pred, bin_label)
                    prec += precision(bin_pred, bin_label)
                    
                    ##Se calculan las diferentes métricas
            avg_acc = acc / len(test_dataloader)
            avg_rec = rec / len(test_dataloader)
            avg_prec = prec / len(test_dataloader)

        print(f"Accuracy: {avg_acc}. Recall: {avg_rec}. Precision: {avg_prec}")
        accs.append(avg_acc)
        recs.append(avg_rec)
        precs.append(avg_prec)

    return np.mean(accs), np.mean(recs), np.mean(precs), np.std(accs), np.std(recs), np.std(precs)

## SIN TL CON OC

In [4]:

def accuracy(outputs, labels):
    preds = outputs > 0.5  # Si la salida es mayor que 0.5, consideramos que es positiva
    correct = (preds == labels.byte())  # Compara las predicciones con las etiquetas reales
    acc = correct.float().sum() / labels.size(0)  # Calcula la precisión
    return acc.item()

def recall(outputs, labels):
    preds = outputs > 0.5

    preds = preds.cpu()
    labels = labels.cpu()
    TP = sum((preds == 1) & (labels == 1))
    FN = sum((preds == 0) & (labels == 1))

    recall_score = TP / (TP + FN) if (TP + FN) > 0 else 0
    return recall_score

def precision(outputs, labels):
    preds = outputs > 0.5

    preds = preds.cpu()
    labels = labels.cpu()

    TP = sum((preds == 1) & (labels == 1))
    FP = sum((preds == 1) & (labels == 0))

    precision_score = TP / (TP + FP) if (TP + FP) > 0 else 0
    return precision_score



def prueba_k_fold_resnet_no_pre_oc(dataset, num_folds_outer, num_epochs, op, batch_size, lr, data_aug, ini, random_seed, seeds_tr_split):
    # Transformaciones para preprocesar las imágenes
    if data_aug:
        transform_t = v2.Compose([
            v2.Resize((224, 224)),
            # v2.RandomResizedCrop(224),
            v2.RandomHorizontalFlip(),
            v2.RandomVerticalFlip(),
            # v2.ElasticTransform(alpha=250.0),
            # v2.RandomRotation(45),
            # transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.2),
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

        transform_v = v2.Compose([
            v2.Resize((224, 224)),
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])
    else:
        transform_t = v2.Compose([
            v2.Resize((224, 224)),
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

        transform_v = v2.Compose([
            v2.Resize((224, 224)),
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

    # Modelo personalizado basado en ResNet
    class CustomResNet(nn.Module):
        def __init__(self, num_classes_binary=1):
            super(CustomResNet, self).__init__()
            resnet = models.resnet18(weights='IMAGENET1K_V1')
            # Extraer las capas de la red ResNet, excluyendo la capa de clasificación
            self.features = nn.Sequential(*list(resnet.children())[:-1])


            # Personalizar la capa de salida para tus necesidades
            self.fc_binary = nn.Linear(512, num_classes_binary)

            if (ini== "G"):
                # Inicialización de Glorot para las capas fully connected
                self.apply(self.init_weights_glorot)
            elif(ini == "H"):
                self.apply(self.init_weights_he)
            elif (ini== "N"):
                self.apply(self.init_weights_normal)
            elif(ini == "U"):
                self.apply(self.init_weights_uniform)
            elif (ini== "O"):
                self.apply(self.init_weights_orthogonal)
            elif(ini == "Z"):
                self.apply(self.init_weights_zeros)
            elif(ini == "ON"):
                self.apply(self.init_weights_ones)

        def init_weights_glorot(self, m):
            if isinstance(m, nn.Linear):
                init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    init.constant_(m.bias, 0)

        def init_weights_he(self, m):
            if isinstance(m, nn.Linear):
                init.kaiming_uniform_(m.weight, mode='fan_in', nonlinearity='relu')
                if m.bias is not None:
                    init.constant_(m.bias, 0)
        def init_weights_normal(self, m):
            if isinstance(m, nn.Linear):
                init.normal_(m.weight, mean=0, std=0.01)  # Puedes ajustar la media y la desviación estándar según tus necesidades
                if m.bias is not None:
                    init.constant_(m.bias, 0)

        def init_weights_uniform(self, m):
            if isinstance(m, nn.Linear):
                init.uniform_(m.weight, a=-0.1, b=0.1)  # Puedes ajustar los límites inferior y superior según tus necesidades
                if m.bias is not None:
                    init.constant_(m.bias, 0)

        def init_weights_orthogonal(self, m):
            if isinstance(m, nn.Linear):
                init.orthogonal_(m.weight)  # Puedes ajustar el parámetro 'gain' según tus necesidades
                if m.bias is not None:
                    init.constant_(m.bias, 0)

        def init_weights_zeros(self, m):
            if isinstance(m, nn.Linear):
                init.zeros_(m.weight)
                if m.bias is not None:
                    init.zeros_(m.bias)

        def init_weights_ones(self, m):
            if isinstance(m, nn.Linear):
                init.ones_(m.weight)
                if m.bias is not None:
                    init.ones_(m.bias)

        def forward(self, x):
            x = self.features(x)
            x = x.view(x.size(0), -1)


            # Salida para la variable binaria
            out_binary = self.fc_binary(x)

            return out_binary

    # Definir una clase para el conjunto de datos personalizado
    ##adaptar esto para meter unas rutsa y obtener dataos y etiquetas
    class CustomDataset(Dataset):
        def __init__(self, dataframe, transform=None):
            self.data = dataframe
            self.transform = transform

        def __len__(self):
            return len(self.data)

        def __getitem__(self, idx):
            img_path = self.data.iloc[idx, 0]  # Ruta de la imagen en la primera columna
            image = Image.open(img_path).convert('RGB')
            # Etiquetas: una variable binaria
            label_binary = torch.tensor(self.data.iloc[idx, 1], dtype=torch.float32)

            if self.transform:
                image = self.transform(image)

            return image, label_binary

    #####5 fold CV
    from sklearn.model_selection import StratifiedKFold

    # Definir el número de folds
    criterion_binary = nn.BCEWithLogitsLoss()
    kf_outer = StratifiedKFold(n_splits=num_folds_outer, shuffle=True, random_state=random_seed)
    ##No será necesario el inner porque la partición será fija en 80%/20%
    # kf_inner = StratifiedKFold(n_splits=num_folds_inner, shuffle=True, random_state=14)
    # Listas para almacenar las pérdidas de entrenamiento y validación de cada fold
    accs = []
    recs = []
    precs = []
    

    
    # Aqui se itera sobre los folds outer, es decir, uno de los folds se usará para test, ya que el modelo no lo verá durante el entrenamiento.

    ##Por otro lado, la parte de train, se verá otra vez sometida a un kfold, 

    for fold, (train_index, test_index) in enumerate(kf_outer.split(np.zeros(len(dataset)),dataset['Etiqueta'])):
        print(f"Fold {fold + 1}")

        # Dividir el conjunto de datos en entrenamiento y validación para este fold
        train_data_fold = dataset.iloc[train_index]
        test_data_fold = dataset.iloc[test_index]

        ##Aqui ya tengo train y test por lo que, antes del bucle para las épocas, se hará la partición con sklearn train test split para validación y train
        ##como se hace dentro del bucle para el outer CV, se hará una vez por fold, que es justo lo que queremos para promediar su rendimiento en un conjunto de test
        ##el cual no se ha usado en ningún momento para entrenar
        train_data_inner, val_data_inner = train_test_split(train_data_fold, test_size=0.2, random_state=seeds_tr_split[fold-1], stratify = train_data_fold['Etiqueta'])
        # Crear instancias del conjunto de datos personalizado para este fold
        train_dataset_fold = CustomDataset(dataframe=train_data_inner, transform=transform_t)
        val_dataset_fold = CustomDataset(dataframe=val_data_inner, transform=transform_v)

        test_dataset = CustomDataset(dataframe=test_data_fold, transform=transform_v)
        # Crear DataLoader para cargar los datos en lotes
        train_loader_fold_in = DataLoader(dataset=train_dataset_fold, batch_size=batch_size, shuffle=True)
        val_loader_fold_in = DataLoader(dataset=val_dataset_fold, batch_size=batch_size, shuffle=False)

        ##Aqui el batch size será de la longitud del test_dataset
        ##como se usa con model eval daría un poco igual el batch size, como en validacion
        test_dataloader = DataLoader(dataset=test_dataset, batch_size=len(test_data_fold), shuffle=False)
       
        # Re-inicializar el modelo para cada fold
        model = CustomResNet(num_classes_binary=1).to(device)
        optimizer = op(model.parameters(), lr = lr)
        scheduler = optim.lr_scheduler.OneCycleLR(optimizer, max_lr=lr, steps_per_epoch=len(train_loader_fold_in), epochs = num_epochs)
        
        # Entrenamiento y validación para este fold
        
        val_loss_best = -1
        best_epoc = -1
        best_model = None
        for epoch in range(num_epochs):
            # Entrenamiento del modelo
            # Fase de entrenamiento
            model.train()
            loss_total_acumulada = 0.0
            
            for images, bin_label in train_loader_fold_in:
                # for i in images:
                #     print(i.shape)
                # images_grid = utils.make_grid(images, nrow=8)  # Hacer una cuadrícula de imágenes
                # plt.figure(figsize=(10, 10))
                # plt.imshow(np.transpose(images_grid, (1, 2, 0)))  # Transponer las dimensiones para mostrar correctamente
                # plt.axis('off')
                # plt.show()
                # break  # Salir después de mostrar el primer lote
                # Reiniciar gradientes
                # Reiniciar gradientes
                images, bin_label = images.to(device), bin_label.to(device)
                optimizer.zero_grad()
                # Forward pass
                bin_out = model(images)
                # Calcular pérdida para las variables continuas
                # Calcular pérdida para la variable binaria
                lossT_bin = criterion_binary(bin_out.squeeze(1), bin_label)
                # Pérdida total
                loss_total = lossT_bin
                loss_total_acumulada += loss_total
                # Backward pass y optimización
                loss_total.backward()
                optimizer.step()
                scheduler.step()
                
            # Validación del modelo
            model.eval()
            val_loss = 0.0
            
            with torch.no_grad():
                for images, bin_label in val_loader_fold_in:
                    # Reiniciar gradientes
                    images,  bin_label = images.to(device),  bin_label.to(device)
                    ##print(len(images)) aqui se hace con 32 imagenes
                    # Forward pass en el conjunto de validación
                    bin_out = model(images)
                    # Calcular pérdida para la variable binaria
                    lossV_bin = criterion_binary(bin_out.squeeze(1), bin_label)
                    # Sumar las pérdidas del conjunto de validación
                    val_loss += lossV_bin.item()
                    

            avg_loss_total_acumulada = loss_total_acumulada / len(train_loader_fold_in)
            
            avg_val_loss = val_loss / len(val_loader_fold_in)
            if(val_loss_best == -1):
                val_loss_best = avg_val_loss
                best_model = copy.deepcopy(model)
                best_epoc = epoch
            else:
                if (avg_val_loss < val_loss_best):
                    val_loss_best = avg_val_loss
                    best_model = copy.deepcopy(model)
                    best_epoc = epoch

            
        ####Aqui es justo antes del cambio de fold, ya tenemos nuestro mejor modelo en base al loss y además tenemos su valor, lo sacamos por pantalla
        print(f"Best epoc: {best_epoc}. Best val loss: {val_loss_best}")

        ##Ahora llegaría el turno de hacer la predicción en test, tanto precision, como recall como accuracy
        acc = 0
        rec = 0
        prec = 0
        best_model.eval()

        with torch.no_grad():
            for images, bin_label in test_dataloader:
                   
                    images, bin_label = images.to(device), bin_label.to(device)
                    
                    bin_out = best_model(images)
                    bin_pred = torch.sigmoid(bin_out.squeeze())
    
                    acc += accuracy(bin_pred, bin_label)
                    rec += recall(bin_pred, bin_label)
                    prec += precision(bin_pred, bin_label)
                    
                    ##Se calculan las diferentes métricas
            avg_acc = acc / len(test_dataloader)
            avg_rec = rec / len(test_dataloader)
            avg_prec = prec / len(test_dataloader)

        print(f"Accuracy: {avg_acc}. Recall: {avg_rec}. Precision: {avg_prec}")
        accs.append(avg_acc)
        recs.append(avg_rec)
        precs.append(avg_prec)

    return np.mean(accs), np.mean(recs), np.mean(precs), np.std(accs), np.std(recs), np.std(precs)

## FUNCION CON TL SIN OC

In [4]:

def accuracy(outputs, labels):
    preds = outputs > 0.5  # Si la salida es mayor que 0.5, consideramos que es positiva
    correct = (preds == labels.byte())  # Compara las predicciones con las etiquetas reales
    acc = correct.float().sum() / labels.size(0)  # Calcula la precisión
    return acc.item()

def recall(outputs, labels):
    preds = outputs > 0.5

    preds = preds.cpu()
    labels = labels.cpu()
    TP = sum((preds == 1) & (labels == 1))
    FN = sum((preds == 0) & (labels == 1))

    recall_score = TP / (TP + FN) if (TP + FN) > 0 else 0
    return recall_score

def precision(outputs, labels):
    preds = outputs > 0.5

    preds = preds.cpu()
    labels = labels.cpu()

    TP = sum((preds == 1) & (labels == 1))
    FP = sum((preds == 1) & (labels == 0))

    precision_score = TP / (TP + FP) if (TP + FP) > 0 else 0
    return precision_score



def prueba_k_fold_resnet_pre(dataset, num_folds_outer, num_epochs_freeze, num_epochs_all, op, batch_size, lr, data_aug, ini, random_seed, seeds_tr_split):
    # Transformaciones para preprocesar las imágenes
    if data_aug:
        transform_t = v2.Compose([
            v2.Resize((224, 224)),
            # v2.RandomResizedCrop(224),
            v2.RandomHorizontalFlip(),
            v2.RandomVerticalFlip(),
            # v2.ElasticTransform(alpha=250.0),
            # v2.RandomRotation(45),
            # transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.2),
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

        transform_v = v2.Compose([
            v2.Resize((224, 224)),
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])
    else:
        transform_t = v2.Compose([
            v2.Resize((224, 224)),
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

        transform_v = v2.Compose([
            v2.Resize((224, 224)),
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

    # Modelo personalizado basado en ResNet
    class CustomResNet(nn.Module):
        def __init__(self, num_classes_binary=1):
            super(CustomResNet, self).__init__()
            resnet = models.resnet18(weights='IMAGENET1K_V1')
            # Extraer las capas de la red ResNet, excluyendo la capa de clasificación
            self.features = nn.Sequential(*list(resnet.children())[:-1])

            # Congelar capas preentrenadas
            for param in self.features.parameters():
                param.requires_grad = False

            # Personalizar la capa de salida para tus necesidades
            self.fc_binary = nn.Linear(512, num_classes_binary)

            if (ini== "G"):
                # Inicialización de Glorot para las capas fully connected
                self.apply(self.init_weights_glorot)
            elif(ini == "H"):
                self.apply(self.init_weights_he)
            elif (ini== "N"):
                self.apply(self.init_weights_normal)
            elif(ini == "U"):
                self.apply(self.init_weights_uniform)
            elif (ini== "O"):
                self.apply(self.init_weights_orthogonal)
            elif(ini == "Z"):
                self.apply(self.init_weights_zeros)
            elif(ini == "ON"):
                self.apply(self.init_weights_ones)

        def init_weights_glorot(self, m):
            if isinstance(m, nn.Linear):
                init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    init.constant_(m.bias, 0)

        def init_weights_he(self, m):
            if isinstance(m, nn.Linear):
                init.kaiming_uniform_(m.weight, mode='fan_in', nonlinearity='relu')
                if m.bias is not None:
                    init.constant_(m.bias, 0)
        def init_weights_normal(self, m):
            if isinstance(m, nn.Linear):
                init.normal_(m.weight, mean=0, std=0.01)  # Puedes ajustar la media y la desviación estándar según tus necesidades
                if m.bias is not None:
                    init.constant_(m.bias, 0)

        def init_weights_uniform(self, m):
            if isinstance(m, nn.Linear):
                init.uniform_(m.weight, a=-0.1, b=0.1)  # Puedes ajustar los límites inferior y superior según tus necesidades
                if m.bias is not None:
                    init.constant_(m.bias, 0)

        def init_weights_orthogonal(self, m):
            if isinstance(m, nn.Linear):
                init.orthogonal_(m.weight)  # Puedes ajustar el parámetro 'gain' según tus necesidades
                if m.bias is not None:
                    init.constant_(m.bias, 0)

        def init_weights_zeros(self, m):
            if isinstance(m, nn.Linear):
                init.zeros_(m.weight)
                if m.bias is not None:
                    init.zeros_(m.bias)

        def init_weights_ones(self, m):
            if isinstance(m, nn.Linear):
                init.ones_(m.weight)
                if m.bias is not None:
                    init.ones_(m.bias)

        def forward(self, x):
            x = self.features(x)
            x = x.view(x.size(0), -1)


            # Salida para la variable binaria
            out_binary = self.fc_binary(x)

            return out_binary

    # Definir una clase para el conjunto de datos personalizado
    ##adaptar esto para meter unas rutsa y obtener dataos y etiquetas
    class CustomDataset(Dataset):
        def __init__(self, dataframe, transform=None):
            self.data = dataframe
            self.transform = transform

        def __len__(self):
            return len(self.data)

        def __getitem__(self, idx):
            img_path = self.data.iloc[idx, 0]  # Ruta de la imagen en la primera columna
            image = Image.open(img_path).convert('RGB')
            # Etiquetas: una variable binaria
            label_binary = torch.tensor(self.data.iloc[idx, 1], dtype=torch.float32)

            if self.transform:
                image = self.transform(image)

            return image, label_binary

    #####5 fold CV
    from sklearn.model_selection import StratifiedKFold

    # Definir el número de folds
    criterion_binary = nn.BCEWithLogitsLoss()
    kf_outer = StratifiedKFold(n_splits=num_folds_outer, shuffle=True, random_state=random_seed)
    ##No será necesario el inner porque la partición será fija en 80%/20%
    # kf_inner = StratifiedKFold(n_splits=num_folds_inner, shuffle=True, random_state=14)
    # Listas para almacenar las pérdidas de entrenamiento y validación de cada fold
    accs = []
    recs = []
    precs = []
    

    
    # Aqui se itera sobre los folds outer, es decir, uno de los folds se usará para test, ya que el modelo no lo verá durante el entrenamiento.

    ##Por otro lado, la parte de train, se verá otra vez sometida a un kfold, 
    
    for fold, (train_index, test_index) in enumerate(kf_outer.split(np.zeros(len(dataset)),dataset['Etiqueta'])):
        print(f"Fold {fold + 1}")

        # Dividir el conjunto de datos en entrenamiento y validación para este fold
        train_data_fold = dataset.iloc[train_index]
        test_data_fold = dataset.iloc[test_index]

        ##Aqui ya tengo train y test por lo que, antes del bucle para las épocas, se hará la partición con sklearn train test split para validación y train
        ##como se hace dentro del bucle para el outer CV, se hará una vez por fold, que es justo lo que queremos para promediar su rendimiento en un conjunto de test
        ##el cual no se ha usado en ningún momento para entrenar
        
        train_data_inner, val_data_inner = train_test_split(train_data_fold, test_size=0.2, random_state=seeds_tr_split[fold-1], stratify=train_data_fold['Etiqueta'])
        
        # Crear instancias del conjunto de datos personalizado para este fold
        train_dataset_fold = CustomDataset(dataframe=train_data_inner, transform=transform_t)
        val_dataset_fold = CustomDataset(dataframe=val_data_inner, transform=transform_v)

        test_dataset = CustomDataset(dataframe=test_data_fold, transform=transform_v)
        # Crear DataLoader para cargar los datos en lotes
        train_loader_fold_in = DataLoader(dataset=train_dataset_fold, batch_size=batch_size, shuffle=True)
        val_loader_fold_in = DataLoader(dataset=val_dataset_fold, batch_size=batch_size, shuffle=False)

        ##Aqui el batch size será de la longitud del test_dataset
        ##como se usa con model eval daría un poco igual el batch size, como en validacion
        test_dataloader = DataLoader(dataset=test_dataset, batch_size=len(test_data_fold), shuffle=False)
       
        # Re-inicializar el modelo para cada fold
        model = CustomResNet(num_classes_binary=1).to(device)
        optimizer = op(model.parameters(), lr = lr)
        optimizer2 = op(model.fc_binary.parameters(), lr = lr)
        # Entrenamiento y validación para este fold
        
        val_loss_best = -1
        best_epoc = -1
        best_model = None
        for epoch in range(num_epochs_freeze):
            # Entrenamiento del modelo
            # Fase de entrenamiento
            model.train()
            loss_total_acumulada = 0.0
            
            for images, bin_label in train_loader_fold_in:
                # for i in images:
                #     print(i.shape)
                # images_grid = utils.make_grid(images, nrow=8)  # Hacer una cuadrícula de imágenes
                # plt.figure(figsize=(10, 10))
                # plt.imshow(np.transpose(images_grid, (1, 2, 0)))  # Transponer las dimensiones para mostrar correctamente
                # plt.axis('off')
                # plt.show()
                # break  # Salir después de mostrar el primer lote
                # Reiniciar gradientes
                # Reiniciar gradientes
                images, bin_label = images.to(device), bin_label.to(device)
                optimizer2.zero_grad()
                # Forward pass
                bin_out = model(images)
                # Calcular pérdida para las variables continuas
                # Calcular pérdida para la variable binaria
                lossT_bin = criterion_binary(bin_out.squeeze(1), bin_label)
                # Pérdida total
                loss_total = lossT_bin
                loss_total_acumulada += loss_total
                # Backward pass y optimización
                loss_total.backward()
                optimizer2.step()
                
            # Validación del modelo
            model.eval()
            val_loss = 0.0
            
            with torch.no_grad():
                for images, bin_label in val_loader_fold_in:
                    # Reiniciar gradientes
                    images,  bin_label = images.to(device),  bin_label.to(device)
                    ##print(len(images)) aqui se hace con 32 imagenes
                    # Forward pass en el conjunto de validación
                    bin_out = model(images)
                    # Calcular pérdida para la variable binaria
                    lossV_bin = criterion_binary(bin_out.squeeze(1), bin_label)
                    # Sumar las pérdidas del conjunto de validación
                    val_loss += lossV_bin.item()
                    

            avg_loss_total_acumulada = loss_total_acumulada / len(train_loader_fold_in)
            
            avg_val_loss = val_loss / len(val_loader_fold_in)
            if(val_loss_best == -1):
                val_loss_best = avg_val_loss
                best_model = copy.deepcopy(model)
                best_epoc = epoch
            else:
                if (avg_val_loss < val_loss_best):
                    val_loss_best = avg_val_loss
                    best_model = copy.deepcopy(model)
                    best_epoc = epoch
            # print(f'Epoch [{epoch+1}/{num_epochs}], Training Loss: {avg_loss_total_acumulada.item()}, Validation Loss: {avg_val_loss}')

            ##Se descongelan todas
        for param in model.features.parameters():
            param.requires_grad = True
        for epoch in range(num_epochs_freeze, num_epochs_freeze + num_epochs_all):
            # Entrenamiento del modelo
            # Fase de entrenamiento
            model.train()
            loss_total_acumulada = 0.0
            
            for images, bin_label in train_loader_fold_in:
                # Reiniciar gradientes
                images, bin_label = images.to(device), bin_label.to(device)
                optimizer.zero_grad()
                # Forward pass
                bin_out = model(images)
                # Calcular pérdida para las variables continuas
                # Calcular pérdida para la variable binaria
                lossT_bin = criterion_binary(bin_out.squeeze(1), bin_label)
                # Pérdida total
                loss_total = lossT_bin
                loss_total_acumulada += loss_total
                # Backward pass y optimización
                loss_total.backward()
                optimizer.step()
                
            # Validación del modelo
            model.eval()
            val_loss = 0.0
            
            with torch.no_grad():
                for images, bin_label in val_loader_fold_in:
                    images,  bin_label = images.to(device),  bin_label.to(device)
                    ##print(len(images)) aqui se hace con 32 imagenes
                    # Forward pass en el conjunto de validación
                    bin_out = model(images)
                    # Calcular pérdida para la variable binaria
                    lossV_bin = criterion_binary(bin_out.squeeze(1), bin_label)
                    # Sumar las pérdidas del conjunto de validación
                    val_loss += lossV_bin.item()
                    ##La cosa sería comparar cada loss con la anterior e ir quedándote con la más baja, guardando el modelo o algo por el estilo
                    ##también se podría hacer por accuracy + recall, o hacer una ponderación de los tres
                    ###ALGO COMO
                    ################### modelo_mejor = modelo si se cumplen las características, posteriormente se hará predicción en test.
                    
                    
            avg_loss_total_acumulada = loss_total_acumulada / len(train_loader_fold_in)
            
            
            avg_val_loss = val_loss / len(val_loader_fold_in)
            if(val_loss_best == -1):
                val_loss_best = avg_val_loss
                best_model = copy.deepcopy(model)
                best_epoc = epoch
            else:
                if (avg_val_loss < val_loss_best):
                    val_loss_best = avg_val_loss
                    best_model = copy.deepcopy(model)
                    best_epoc = epoch
            
        
        ####Aqui es justo antes del cambio de fold, ya tenemos nuestro mejor modelo en base al loss y además tenemos su valor, lo sacamos por pantalla
        print(f"Best epoc: {best_epoc}. Best val loss: {val_loss_best}")

        ##Ahora llegaría el turno de hacer la predicción en test, tanto precision, como recall como accuracy
        acc = 0
        rec = 0
        prec = 0
        best_model.eval()

        with torch.no_grad():
            for images, bin_label in test_dataloader:
                   
                    images, bin_label = images.to(device), bin_label.to(device)
                    
                    bin_out = best_model(images)
                    bin_pred = torch.sigmoid(bin_out.squeeze())
    
                    acc += accuracy(bin_pred, bin_label)
                    rec += recall(bin_pred, bin_label)
                    prec += precision(bin_pred, bin_label)
                    
                    ##Se calculan las diferentes métricas
            avg_acc = acc / len(test_dataloader)
            avg_rec = rec / len(test_dataloader)
            avg_prec = prec / len(test_dataloader)

        print(f"Accuracy: {avg_acc}. Recall: {avg_rec}. Precision: {avg_prec}")
        accs.append(avg_acc)
        recs.append(avg_rec)
        precs.append(avg_prec)

        

    return np.mean(accs), np.mean(recs), np.mean(precs), np.std(accs), np.std(recs), np.std(precs)

## FUNCIÓN CON TL Y OC

In [5]:

def accuracy(outputs, labels):
    preds = outputs > 0.5  # Si la salida es mayor que 0.5, consideramos que es positiva
    correct = (preds == labels.byte())  # Compara las predicciones con las etiquetas reales
    acc = correct.float().sum() / labels.size(0)  # Calcula la precisión
    return acc.item()

def recall(outputs, labels):
    preds = outputs > 0.5

    preds = preds.cpu()
    labels = labels.cpu()
    TP = sum((preds == 1) & (labels == 1))
    FN = sum((preds == 0) & (labels == 1))

    recall_score = TP / (TP + FN) if (TP + FN) > 0 else 0
    return recall_score

def precision(outputs, labels):
    preds = outputs > 0.5

    preds = preds.cpu()
    labels = labels.cpu()

    TP = sum((preds == 1) & (labels == 1))
    FP = sum((preds == 1) & (labels == 0))

    precision_score = TP / (TP + FP) if (TP + FP) > 0 else 0
    return precision_score



def prueba_k_fold_resnet_pre_oc(dataset, num_folds_outer, num_epochs_freeze, num_epochs_all, op, batch_size, lr, data_aug, ini, random_seed, seeds_tr_split):
    # Transformaciones para preprocesar las imágenes
    if data_aug:
        transform_t = v2.Compose([
            v2.Resize((224, 224)),
            # v2.RandomResizedCrop(224),
            v2.RandomHorizontalFlip(),
            v2.RandomVerticalFlip(),
            # v2.ElasticTransform(alpha=250.0),
            # v2.RandomRotation(45),
            # transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.2),
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

        transform_v = v2.Compose([
            v2.Resize((224, 224)),
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])
    else:
        transform_t = v2.Compose([
            v2.Resize((224, 224)),
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

        transform_v = v2.Compose([
            v2.Resize((224, 224)),
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

    # Modelo personalizado basado en ResNet
    class CustomResNet(nn.Module):
        def __init__(self, num_classes_binary=1):
            super(CustomResNet, self).__init__()
            resnet = models.resnet18(weights='IMAGENET1K_V1')
            # Extraer las capas de la red ResNet, excluyendo la capa de clasificación
            self.features = nn.Sequential(*list(resnet.children())[:-1])

            # Congelar capas preentrenadas
            for param in self.features.parameters():
                param.requires_grad = False

            # Personalizar la capa de salida para tus necesidades
            self.fc_binary = nn.Linear(512, num_classes_binary)

            if (ini== "G"):
                # Inicialización de Glorot para las capas fully connected
                self.apply(self.init_weights_glorot)
            elif(ini == "H"):
                self.apply(self.init_weights_he)
            elif (ini== "N"):
                self.apply(self.init_weights_normal)
            elif(ini == "U"):
                self.apply(self.init_weights_uniform)
            elif (ini== "O"):
                self.apply(self.init_weights_orthogonal)
            elif(ini == "Z"):
                self.apply(self.init_weights_zeros)
            elif(ini == "ON"):
                self.apply(self.init_weights_ones)

        def init_weights_glorot(self, m):
            if isinstance(m, nn.Linear):
                init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    init.constant_(m.bias, 0)

        def init_weights_he(self, m):
            if isinstance(m, nn.Linear):
                init.kaiming_uniform_(m.weight, mode='fan_in', nonlinearity='relu')
                if m.bias is not None:
                    init.constant_(m.bias, 0)
        def init_weights_normal(self, m):
            if isinstance(m, nn.Linear):
                init.normal_(m.weight, mean=0, std=0.01)  # Puedes ajustar la media y la desviación estándar según tus necesidades
                if m.bias is not None:
                    init.constant_(m.bias, 0)

        def init_weights_uniform(self, m):
            if isinstance(m, nn.Linear):
                init.uniform_(m.weight, a=-0.1, b=0.1)  # Puedes ajustar los límites inferior y superior según tus necesidades
                if m.bias is not None:
                    init.constant_(m.bias, 0)

        def init_weights_orthogonal(self, m):
            if isinstance(m, nn.Linear):
                init.orthogonal_(m.weight)  # Puedes ajustar el parámetro 'gain' según tus necesidades
                if m.bias is not None:
                    init.constant_(m.bias, 0)

        def init_weights_zeros(self, m):
            if isinstance(m, nn.Linear):
                init.zeros_(m.weight)
                if m.bias is not None:
                    init.zeros_(m.bias)

        def init_weights_ones(self, m):
            if isinstance(m, nn.Linear):
                init.ones_(m.weight)
                if m.bias is not None:
                    init.ones_(m.bias)

        def forward(self, x):
            x = self.features(x)
            x = x.view(x.size(0), -1)


            # Salida para la variable binaria
            out_binary = self.fc_binary(x)

            return out_binary

    # Definir una clase para el conjunto de datos personalizado
    ##adaptar esto para meter unas rutsa y obtener dataos y etiquetas
    class CustomDataset(Dataset):
        def __init__(self, dataframe, transform=None):
            self.data = dataframe
            self.transform = transform

        def __len__(self):
            return len(self.data)

        def __getitem__(self, idx):
            img_path = self.data.iloc[idx, 0]  # Ruta de la imagen en la primera columna
            image = Image.open(img_path).convert('RGB')
            # Etiquetas: una variable binaria
            label_binary = torch.tensor(self.data.iloc[idx, 1], dtype=torch.float32)

            if self.transform:
                image = self.transform(image)

            return image, label_binary

    #####5 fold CV
    from sklearn.model_selection import StratifiedKFold

    # Definir el número de folds
    criterion_binary = nn.BCEWithLogitsLoss()
    kf_outer = StratifiedKFold(n_splits=num_folds_outer, shuffle=True, random_state=random_seed)
    ##No será necesario el inner porque la partición será fija en 80%/20%
    # kf_inner = StratifiedKFold(n_splits=num_folds_inner, shuffle=True, random_state=14)
    # Listas para almacenar las pérdidas de entrenamiento y validación de cada fold
    accs = []
    recs = []
    precs = []
    

    
    # Aqui se itera sobre los folds outer, es decir, uno de los folds se usará para test, ya que el modelo no lo verá durante el entrenamiento.

    ##Por otro lado, la parte de train, se verá otra vez sometida a un kfold, 
    
    for fold, (train_index, test_index) in enumerate(kf_outer.split(np.zeros(len(dataset)),dataset['Etiqueta'])):
        print(f"Fold {fold + 1}")

        # Dividir el conjunto de datos en entrenamiento y validación para este fold
        train_data_fold = dataset.iloc[train_index]
        test_data_fold = dataset.iloc[test_index]

        ##Aqui ya tengo train y test por lo que, antes del bucle para las épocas, se hará la partición con sklearn train test split para validación y train
        ##como se hace dentro del bucle para el outer CV, se hará una vez por fold, que es justo lo que queremos para promediar su rendimiento en un conjunto de test
        ##el cual no se ha usado en ningún momento para entrenar
        
        train_data_inner, val_data_inner = train_test_split(train_data_fold, test_size=0.2, random_state=seeds_tr_split[fold-1], stratify=train_data_fold['Etiqueta'])
        
        # Crear instancias del conjunto de datos personalizado para este fold
        train_dataset_fold = CustomDataset(dataframe=train_data_inner, transform=transform_t)
        val_dataset_fold = CustomDataset(dataframe=val_data_inner, transform=transform_v)

        test_dataset = CustomDataset(dataframe=test_data_fold, transform=transform_v)
        # Crear DataLoader para cargar los datos en lotes
        train_loader_fold_in = DataLoader(dataset=train_dataset_fold, batch_size=batch_size, shuffle=True)
        val_loader_fold_in = DataLoader(dataset=val_dataset_fold, batch_size=batch_size, shuffle=False)

        ##Aqui el batch size será de la longitud del test_dataset
        ##como se usa con model eval daría un poco igual el batch size, como en validacion
        test_dataloader = DataLoader(dataset=test_dataset, batch_size=len(test_data_fold), shuffle=False)
       
        # Re-inicializar el modelo para cada fold
        model = CustomResNet(num_classes_binary=1).to(device)
        optimizer = op(model.parameters(), lr = lr)
        optimizer2 = op(model.fc_binary.parameters(), lr = lr)

        total = len(optimizer.param_groups)
        # print(total)
        scheduler1 = optim.lr_scheduler.OneCycleLR(optimizer2, max_lr=lr, steps_per_epoch=len(train_loader_fold_in), epochs=num_epochs_freeze)
        # Definir el objeto slice
        lr /= 2
        # lr_slice = slice(lr/100, lr)
        max_lrs = np.linspace(lr/100, lr, total)
        # print(lr_slice.start)
        # print(lr_slice.stop)
        max_lrs = max_lrs.tolist()
        scheduler2 = optim.lr_scheduler.OneCycleLR(optimizer, max_lr=max_lrs, steps_per_epoch=len(train_loader_fold_in), epochs=num_epochs_all)
        # Entrenamiento y validación para este fold
        
        val_loss_best = -1
        best_epoc = -1
        best_model = None
        for epoch in range(num_epochs_freeze):
            # Entrenamiento del modelo
            # Fase de entrenamiento
            model.train()
            loss_total_acumulada = 0.0
            
            for images, bin_label in train_loader_fold_in:
                # for i in images:
                #     print(i.shape)
                # images_grid = utils.make_grid(images, nrow=8)  # Hacer una cuadrícula de imágenes
                # plt.figure(figsize=(10, 10))
                # plt.imshow(np.transpose(images_grid, (1, 2, 0)))  # Transponer las dimensiones para mostrar correctamente
                # plt.axis('off')
                # plt.show()
                # break  # Salir después de mostrar el primer lote
                # Reiniciar gradientes
                # Reiniciar gradientes
                images, bin_label = images.to(device), bin_label.to(device)
                optimizer2.zero_grad()
                # Forward pass
                bin_out = model(images)
                # Calcular pérdida para las variables continuas
                # Calcular pérdida para la variable binaria
                lossT_bin = criterion_binary(bin_out.squeeze(1), bin_label)
                # Pérdida total
                loss_total = lossT_bin
                loss_total_acumulada += loss_total
                # Backward pass y optimización
                loss_total.backward()
                optimizer2.step()
                scheduler1.step()
                
            # Validación del modelo
            model.eval()
            val_loss = 0.0
            
            with torch.no_grad():
                for images, bin_label in val_loader_fold_in:
                    # Reiniciar gradientes
                    images,  bin_label = images.to(device),  bin_label.to(device)
                    ##print(len(images)) aqui se hace con 32 imagenes
                    # Forward pass en el conjunto de validación
                    bin_out = model(images)
                    # Calcular pérdida para la variable binaria
                    lossV_bin = criterion_binary(bin_out.squeeze(1), bin_label)
                    # Sumar las pérdidas del conjunto de validación
                    val_loss += lossV_bin.item()
                    

            avg_loss_total_acumulada = loss_total_acumulada / len(train_loader_fold_in)
            
            avg_val_loss = val_loss / len(val_loader_fold_in)
            if(val_loss_best == -1):
                val_loss_best = avg_val_loss
                best_model = copy.deepcopy(model)
                best_epoc = epoch
            else:
                if (avg_val_loss < val_loss_best):
                    val_loss_best = avg_val_loss
                    best_model = copy.deepcopy(model)
                    best_epoc = epoch
            # print(f'Epoch [{epoch+1}/{num_epochs}], Training Loss: {avg_loss_total_acumulada.item()}, Validation Loss: {avg_val_loss}')

            ##Se descongelan todas
        for param in model.features.parameters():
            param.requires_grad = True
        for epoch in range(num_epochs_freeze, num_epochs_freeze + num_epochs_all):
            # Entrenamiento del modelo
            # Fase de entrenamiento
            model.train()
            loss_total_acumulada = 0.0
            
            for images, bin_label in train_loader_fold_in:
                # Reiniciar gradientes
                images, bin_label = images.to(device), bin_label.to(device)
                optimizer.zero_grad()
                # Forward pass
                bin_out = model(images)
                # Calcular pérdida para las variables continuas
                # Calcular pérdida para la variable binaria
                lossT_bin = criterion_binary(bin_out.squeeze(1), bin_label)
                # Pérdida total
                loss_total = lossT_bin
                loss_total_acumulada += loss_total
                # Backward pass y optimización
                loss_total.backward()
                optimizer.step()
                scheduler2.step()
                
            # Validación del modelo
            model.eval()
            val_loss = 0.0
            
            with torch.no_grad():
                for images, bin_label in val_loader_fold_in:
                    images,  bin_label = images.to(device),  bin_label.to(device)
                    ##print(len(images)) aqui se hace con 32 imagenes
                    # Forward pass en el conjunto de validación
                    bin_out = model(images)
                    # Calcular pérdida para la variable binaria
                    lossV_bin = criterion_binary(bin_out.squeeze(1), bin_label)
                    # Sumar las pérdidas del conjunto de validación
                    val_loss += lossV_bin.item()
                    ##La cosa sería comparar cada loss con la anterior e ir quedándote con la más baja, guardando el modelo o algo por el estilo
                    ##también se podría hacer por accuracy + recall, o hacer una ponderación de los tres
                    ###ALGO COMO
                    ################### modelo_mejor = modelo si se cumplen las características, posteriormente se hará predicción en test.
                    
                    
            avg_loss_total_acumulada = loss_total_acumulada / len(train_loader_fold_in)
            
            
            avg_val_loss = val_loss / len(val_loader_fold_in)
            if(val_loss_best == -1):
                val_loss_best = avg_val_loss
                best_model = copy.deepcopy(model)
                best_epoc = epoch
            else:
                if (avg_val_loss < val_loss_best):
                    val_loss_best = avg_val_loss
                    best_model = copy.deepcopy(model)
                    best_epoc = epoch
            
        
        ####Aqui es justo antes del cambio de fold, ya tenemos nuestro mejor modelo en base al loss y además tenemos su valor, lo sacamos por pantalla
        print(f"Best epoc: {best_epoc}. Best val loss: {val_loss_best}")

        ##Ahora llegaría el turno de hacer la predicción en test, tanto precision, como recall como accuracy
        acc = 0
        rec = 0
        prec = 0
        best_model.eval()

        with torch.no_grad():
            for images, bin_label in test_dataloader:
                   
                    images, bin_label = images.to(device), bin_label.to(device)
                    
                    bin_out = best_model(images)
                    bin_pred = torch.sigmoid(bin_out.squeeze())
    
                    acc += accuracy(bin_pred, bin_label)
                    rec += recall(bin_pred, bin_label)
                    prec += precision(bin_pred, bin_label)
                    
                    ##Se calculan las diferentes métricas
            avg_acc = acc / len(test_dataloader)
            avg_rec = rec / len(test_dataloader)
            avg_prec = prec / len(test_dataloader)

        print(f"Accuracy: {avg_acc}. Recall: {avg_rec}. Precision: {avg_prec}")
        accs.append(avg_acc)
        recs.append(avg_rec)
        precs.append(avg_prec)

        

    return np.mean(accs), np.mean(recs), np.mean(precs), np.std(accs), np.std(recs), np.std(precs)

## PRUEBA SIN TL SIN OC, 100 EJECUCIONES

In [15]:
seeds = random.sample(range(1,10001), 100)
seeds_print = copy.copy(seeds)

seeds_tr = [random.sample(range(1,10001), 5) for _ in range(100)]
seeds_tr_print = copy.copy(seeds_tr)


##para las semillas de train test split, hago una lista de listas con tantas listas como ejecuciones y cada lista coneteniendo k numeros, según los folds que se hagan
def save_study(study, study_name="study"):
    # Crear una carpeta para guardar el estudio si no existe
    study_dir = os.path.join(os.getcwd(), study_name)
    if not os.path.exists(study_dir):
        os.makedirs(study_dir)
    
    # Guardar los resultados del estudio en un archivo CSV
    study_trials = study.trials_dataframe()
    study_trials.to_csv(os.path.join(study_dir, "study_results.csv"), index=False)

def objective(trial):
    # Define los hiperparámetros para optimizar
    # num_epochs_freeze = trial.suggest_categorical('num_epochs_freeze', [1,3,8,10,15])
    # num_epochs_all = trial.suggest_categorical('num_epochs_all', [25,30,35,40,45,50])
    ##todos estos bs salieron en los mejores, así como ambos extremos del lr
    batch_size = trial.suggest_categorical('batch_size', [5,32])
    lr = trial.suggest_float('lr', 1e-5, 1e-4)
    data_aug = trial.suggest_categorical('data_aug', [True,False])
    ini = trial.suggest_categorical('ini', ['G', 'H', 'O', 'None'])
    all = trial.suggest_int('e_all', 35, 50)
    
    # optimizer_name = trial.suggest_categorical('optimizer_name', ['RMSprop', 'Adam', 'SGD'])

    # # Convertir el nombre del optimizador en una clase PyTorch
    # if optimizer_name == 'Adam':
    #     optimizer_class = torch.optim.Adam
    # elif optimizer_name == 'SGD':
    #     optimizer_class = torch.optim.SGD
    # elif optimizer_name == 'RMSprop':
    #     optimizer_class = torch.optim.RMSprop

    # Ejecuta la función de entrenamiento con los hiperparámetros seleccionados
    # avg_accuracy, avg_recall, avg_precision, acc_std, rec_std, prec_std= prueba_k_fold_resnet_pre(df_mosaic, 5, freeze, all, optim.Adam, batch_size, lr, data_aug, ini, seeds[0], seeds_tr[0])
    avg_accuracy, avg_recall, avg_precision, acc_std, rec_std, prec_std= prueba_k_fold_resnet_no_pre(df_mosaic, 5, all, optim.Adam, batch_size, lr, data_aug, ini, seeds[0], seeds_tr[0])
    seeds.pop(0)
    seeds_tr.pop(0)

    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_pre(df_mean_a_2, 5, 5, 35, optim.Adam, batch_size, lr, data_aug, None)
    # avg_l_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre(df_mosaic, 5, all, optim.Adam, batch_size, lr, data_aug, ini, seeds[0], seeds_tr[0])
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre_oc(df_mean_a_2, 5, 40, optim.Adam, batch_size, lr, data_aug, None)

    # Devuelve el valor objetivo (en este caso, la pérdida promedio de validación)
    return avg_accuracy, avg_recall, avg_precision, acc_std, rec_std, prec_std

import optuna
# Supongamos que tienes un dataset llamado 'dataset'
# Realiza una llamada a Optuna para optimizar los hiperparámetros
study = optuna.create_study(directions = ['maximize','maximize','maximize', 'minimize','minimize','minimize'], sampler=None)#None, NSGA por defecto para multiobjetivo, sampler=optuna.samplers.GridSampler(search_space)
print(f"Sampler is {study.sampler.__class__.__name__}")
study.optimize(objective, n_trials=100)#, n_trials=100

# Muestra los mejores hiperparámetros encontrados
print('Best trial:')
trial = study.best_trials
for t in trial:
    print('Value: {}'.format(t.values))
    print('Params: ')
    for key, value in t.params.items():
        print('{}: {}'.format(key, value))

save_study(study, 'estudio_adam')


###hacer tambien lista para las semillas de los train test split en base al numero de folds que haya

[I 2024-04-26 20:45:09,682] A new study created in memory with name: no-name-69cb03bb-7371-4f98-ae62-6e295330d794


Sampler is NSGAIISampler
Fold 1
Best epoc: 8. Best val loss: 0.5714308023452759
Accuracy: 0.8888888955116272. Recall: 1.0. Precision: 0.800000011920929
Fold 2
Best epoc: 34. Best val loss: 0.014915774576365948
Accuracy: 0.7777777910232544. Recall: 0.5. Precision: 1.0
Fold 3
Best epoc: 3. Best val loss: 0.7546722292900085
Accuracy: 0.5555555820465088. Recall: 1.0. Precision: 0.5
Fold 4
Best epoc: 12. Best val loss: 0.1974058449268341
Accuracy: 0.7777777910232544. Recall: 0.75. Precision: 0.75
Fold 5


[I 2024-04-26 20:45:46,879] Trial 0 finished with values: [0.7750000119209289, 0.800000011920929, 0.8100000619888306, 0.11928282804975904, 0.18708285689353943, 0.18547236919403076] and parameters: {'batch_size': 32, 'lr': 8.288403934541641e-05, 'data_aug': True, 'ini': 'O', 'e_all': 35}. 


Best epoc: 15. Best val loss: 0.32479074597358704
Accuracy: 0.875. Recall: 0.75. Precision: 1.0
Fold 1
Best epoc: 21. Best val loss: 0.19156968450988643
Accuracy: 1.0. Recall: 1.0. Precision: 1.0
Fold 2
Best epoc: 27. Best val loss: 0.21221273392438889
Accuracy: 0.7777777910232544. Recall: 0.75. Precision: 0.75
Fold 3
Best epoc: 35. Best val loss: 0.20221026986837387
Accuracy: 0.8888888955116272. Recall: 1.0. Precision: 0.800000011920929
Fold 4
Best epoc: 37. Best val loss: 0.0025207739672623575
Accuracy: 0.6666666865348816. Recall: 0.5. Precision: 0.6666666865348816
Fold 5


[I 2024-04-26 20:46:32,619] Trial 1 finished with values: [0.8666666746139526, 0.8500000238418579, 0.84333336353302, 0.1295767010509663, 0.20000000298023224, 0.1348249763250351] and parameters: {'batch_size': 5, 'lr': 4.12834414731391e-05, 'data_aug': True, 'ini': 'H', 'e_all': 41}. 


Best epoc: 4. Best val loss: 0.4878903180360794
Accuracy: 1.0. Recall: 1.0. Precision: 1.0
Fold 1
Best epoc: 5. Best val loss: 0.19430948048830032
Accuracy: 0.8888888955116272. Recall: 1.0. Precision: 0.800000011920929
Fold 2
Best epoc: 13. Best val loss: 0.06838406971655786
Accuracy: 0.6666666865348816. Recall: 0.5. Precision: 0.6666666865348816
Fold 3
Best epoc: 18. Best val loss: 0.5832975059747696
Accuracy: 0.8888888955116272. Recall: 0.75. Precision: 1.0
Fold 4
Best epoc: 27. Best val loss: 0.09020167030394077
Accuracy: 1.0. Recall: 1.0. Precision: 1.0
Fold 5


... (192 outputs left out)

[I 2024-04-26 21:58:39,472] Trial 98 finished with values: [0.9111111164093018, 0.800000011920929, 1.0, 0.08314793697230627, 0.18708285689353943, 0.0] and parameters: {'batch_size': 5, 'lr': 6.627800688433089e-05, 'data_aug': True, 'ini': 'None', 'e_all': 35}. 


Best epoc: 10. Best val loss: 0.16088131070137024
Accuracy: 1.0. Recall: 1.0. Precision: 1.0
Fold 1
Best epoc: 25. Best val loss: 0.029371706070378423
Accuracy: 0.8888888955116272. Recall: 1.0. Precision: 0.800000011920929
Fold 2
Best epoc: 19. Best val loss: 0.24921785853803158
Accuracy: 0.6666666865348816. Recall: 0.5. Precision: 0.6666666865348816
Fold 3
Best epoc: 6. Best val loss: 0.390630878508091
Accuracy: 0.8888888955116272. Recall: 0.75. Precision: 1.0
Fold 4
Best epoc: 22. Best val loss: 0.36623020470142365
Accuracy: 0.8888888955116272. Recall: 0.75. Precision: 1.0
Fold 5


[I 2024-04-26 21:59:21,700] Trial 99 finished with values: [0.8416666746139526, 0.800000011920929, 0.8533333539962769, 0.08766518219737346, 0.18708285689353943, 0.1292714625597] and parameters: {'batch_size': 5, 'lr': 2.9482914013865476e-05, 'data_aug': True, 'ini': 'None', 'e_all': 37}. 


Best epoc: 9. Best val loss: 0.3410831391811371
Accuracy: 0.875. Recall: 1.0. Precision: 0.800000011920929
Best trial:
Value: [0.9055555582046508, 0.8500000238418579, 0.9600000381469727, 0.09229582022071937, 0.20000000298023224, 0.07999999821186066]
Params: 
batch_size: 32
lr: 4.4085704095909264e-05
data_aug: True
ini: None
e_all: 46
Value: [0.7972222328186035, 0.8500000238418579, 0.800000011920929, 0.038888883590698246, 0.20000000298023224, 0.1632993072271347]
Params: 
batch_size: 5
lr: 4.44793163062718e-05
data_aug: False
ini: G
e_all: 43
Value: [0.8861111164093017, 0.8500000238418579, 0.9199999570846558, 0.07049209348016061, 0.20000000298023224, 0.0979795828461647]
Params: 
batch_size: 5
lr: 9.637788424484944e-05
data_aug: True
ini: O
e_all: 43
Value: [0.9083333373069763, 0.8999999761581421, 0.9199999570846558, 0.046147908675248546, 0.12247449159622192, 0.0979795902967453]
Params: 
batch_size: 5
lr: 6.990675455337156e-05
data_aug: True
ini: O
e_all: 35
Value: [0.9083333373069763, 0.

In [14]:
seeds = random.sample(range(1,10001), 100)
seeds_print = copy.copy(seeds)

seeds_tr = [random.sample(range(1,10001), 5) for _ in range(100)]
seeds_tr_print = copy.copy(seeds_tr)


##para las semillas de train test split, hago una lista de listas con tantas listas como ejecuciones y cada lista coneteniendo k numeros, según los folds que se hagan
def save_study(study, study_name="study"):
    # Crear una carpeta para guardar el estudio si no existe
    study_dir = os.path.join(os.getcwd(), study_name)
    if not os.path.exists(study_dir):
        os.makedirs(study_dir)
    
    # Guardar los resultados del estudio en un archivo CSV
    study_trials = study.trials_dataframe()
    study_trials.to_csv(os.path.join(study_dir, "study_results.csv"), index=False)

def objective(trial):
    # Define los hiperparámetros para optimizar
    # num_epochs_freeze = trial.suggest_categorical('num_epochs_freeze', [1,3,8,10,15])
    # num_epochs_all = trial.suggest_categorical('num_epochs_all', [25,30,35,40,45,50])
    ##todos estos bs salieron en los mejores, así como ambos extremos del lr
    batch_size = trial.suggest_categorical('batch_size', [5,16])
    lr = trial.suggest_float('lr', 0.01/2, 0.01*2)
    data_aug = trial.suggest_categorical('data_aug', [True,False])
    ini = trial.suggest_categorical('ini', ['G', 'H', 'O', 'None'])
    all = trial.suggest_int('e_all', 35, 50)
    
    # optimizer_name = trial.suggest_categorical('optimizer_name', ['RMSprop', 'Adam', 'SGD'])

    # # Convertir el nombre del optimizador en una clase PyTorch
    # if optimizer_name == 'Adam':
    #     optimizer_class = torch.optim.Adam
    # elif optimizer_name == 'SGD':
    #     optimizer_class = torch.optim.SGD
    # elif optimizer_name == 'RMSprop':
    #     optimizer_class = torch.optim.RMSprop

    # Ejecuta la función de entrenamiento con los hiperparámetros seleccionados
    # avg_accuracy, avg_recall, avg_precision, acc_std, rec_std, prec_std= prueba_k_fold_resnet_pre(df_mosaic, 5, freeze, all, optim.Adam, batch_size, lr, data_aug, ini, seeds[0], seeds_tr[0])
    avg_accuracy, avg_recall, avg_precision, acc_std, rec_std, prec_std= prueba_k_fold_resnet_no_pre(df_mosaic, 5, all, optim.SGD, batch_size, lr, data_aug, ini, seeds[0], seeds_tr[0])
    seeds.pop(0)
    seeds_tr.pop(0)

    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_pre(df_mean_a_2, 5, 5, 35, optim.Adam, batch_size, lr, data_aug, None)
    # avg_l_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre(df_mosaic, 5, all, optim.Adam, batch_size, lr, data_aug, ini, seeds[0], seeds_tr[0])
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre_oc(df_mean_a_2, 5, 40, optim.Adam, batch_size, lr, data_aug, None)

    # Devuelve el valor objetivo (en este caso, la pérdida promedio de validación)
    return avg_accuracy, avg_recall, avg_precision, acc_std, rec_std, prec_std

import optuna
# Supongamos que tienes un dataset llamado 'dataset'
# Realiza una llamada a Optuna para optimizar los hiperparámetros
study = optuna.create_study(directions = ['maximize','maximize','maximize', 'minimize','minimize','minimize'], sampler=None)#None, NSGA por defecto para multiobjetivo, sampler=optuna.samplers.GridSampler(search_space)
print(f"Sampler is {study.sampler.__class__.__name__}")
study.optimize(objective, n_trials=100)#, n_trials=100

# Muestra los mejores hiperparámetros encontrados
print('Best trial:')
trial = study.best_trials
for t in trial:
    print('Value: {}'.format(t.values))
    print('Params: ')
    for key, value in t.params.items():
        print('{}: {}'.format(key, value))

save_study(study, 'estudio_sgd')


###hacer tambien lista para las semillas de los train test split en base al numero de folds que haya

[I 2024-04-26 19:37:40,744] A new study created in memory with name: no-name-05fbaf9a-ab26-4efc-a363-f9658f891f01


Sampler is NSGAIISampler
Fold 1
Best epoc: 7. Best val loss: 0.29509061574935913
Accuracy: 0.8888888955116272. Recall: 1.0. Precision: 0.800000011920929
Fold 2
Best epoc: 33. Best val loss: 0.1899844855070114
Accuracy: 0.7777777910232544. Recall: 0.5. Precision: 1.0
Fold 3
Best epoc: 40. Best val loss: 0.3682824373245239
Accuracy: 1.0. Recall: 1.0. Precision: 1.0
Fold 4
Best epoc: 3. Best val loss: 0.6244816780090332
Accuracy: 0.8888888955116272. Recall: 0.75. Precision: 1.0
Fold 5


[I 2024-04-26 19:38:20,414] Trial 0 finished with values: [0.8611111164093017, 0.75, 0.9600000381469727, 0.08958064000486261, 0.22360679507255554, 0.07999999821186066] and parameters: {'batch_size': 16, 'lr': 0.01827458188118181, 'data_aug': True, 'ini': 'H', 'e_all': 46}. 


Best epoc: 6. Best val loss: 0.2994979918003082
Accuracy: 0.75. Recall: 0.5. Precision: 1.0
Fold 1
Best epoc: 3. Best val loss: 0.07760843774303794
Accuracy: 0.7777777910232544. Recall: 0.5. Precision: 1.0
Fold 2
Best epoc: 30. Best val loss: 0.003388178418390453
Accuracy: 1.0. Recall: 1.0. Precision: 1.0
Fold 3
Best epoc: 6. Best val loss: 0.02218719944357872
Accuracy: 0.8888888955116272. Recall: 0.75. Precision: 1.0
Fold 4
Best epoc: 30. Best val loss: 0.005161961307749152
Accuracy: 0.8888888955116272. Recall: 0.75. Precision: 1.0
Fold 5


[I 2024-04-26 19:38:57,694] Trial 1 finished with values: [0.9111111164093018, 0.800000011920929, 1.0, 0.08314793697230627, 0.18708285689353943, 0.0] and parameters: {'batch_size': 5, 'lr': 0.01789978947121082, 'data_aug': True, 'ini': 'O', 'e_all': 35}. 


Best epoc: 21. Best val loss: 0.4344296380877495
Accuracy: 1.0. Recall: 1.0. Precision: 1.0
Fold 1
Best epoc: 33. Best val loss: 0.20526251196861267
Accuracy: 0.6666666865348816. Recall: 0.25. Precision: 1.0
Fold 2
Best epoc: 38. Best val loss: 0.2234654575586319
Accuracy: 0.8888888955116272. Recall: 0.75. Precision: 1.0
Fold 3
Best epoc: 14. Best val loss: 0.4566762447357178
Accuracy: 0.7777777910232544. Recall: 0.75. Precision: 0.75
Fold 4
Best epoc: 8. Best val loss: 0.3343265652656555
Accuracy: 0.8888888955116272. Recall: 1.0. Precision: 0.800000011920929
Fold 5


... (192 outputs left out)

[I 2024-04-26 20:44:36,290] Trial 98 finished with values: [0.8805555582046509, 0.8500000238418579, 0.8933333158493042, 0.13709958548605422, 0.20000000298023224, 0.1372750699520111] and parameters: {'batch_size': 5, 'lr': 0.019524739335771268, 'data_aug': True, 'ini': 'None', 'e_all': 45}. 


Best epoc: 44. Best val loss: 0.00034582329681143165
Accuracy: 0.625. Recall: 0.5. Precision: 0.6666666865348816
Fold 1
Best epoc: 10. Best val loss: 0.3435848355293274
Accuracy: 1.0. Recall: 1.0. Precision: 1.0
Fold 2
Best epoc: 9. Best val loss: 0.6071979999542236
Accuracy: 0.8888888955116272. Recall: 0.75. Precision: 1.0
Fold 3
Best epoc: 9. Best val loss: 0.09614105522632599
Accuracy: 1.0. Recall: 1.0. Precision: 1.0
Fold 4
Best epoc: 21. Best val loss: 0.025311071425676346
Accuracy: 0.8888888955116272. Recall: 0.75. Precision: 1.0
Fold 5


[I 2024-04-26 20:45:09,633] Trial 99 finished with values: [0.9055555582046508, 0.8500000238418579, 0.949999988079071, 0.09229582022071937, 0.12247449159622192, 0.10000000149011612] and parameters: {'batch_size': 16, 'lr': 0.01827458188118181, 'data_aug': True, 'ini': 'H', 'e_all': 38}. 


Best epoc: 19. Best val loss: 0.3127726912498474
Accuracy: 0.75. Recall: 0.75. Precision: 0.75
Best trial:
Value: [0.9111111164093018, 0.800000011920929, 1.0, 0.08314793697230627, 0.18708285689353943, 0.0]
Params: 
batch_size: 5
lr: 0.01789978947121082
data_aug: True
ini: O
e_all: 35
Value: [0.8861111164093017, 0.800000011920929, 0.9600000381469727, 0.005555558204650879, 0.10000000149011612, 0.07999999821186066]
Params: 
batch_size: 5
lr: 0.00638187343671907
data_aug: True
ini: G
e_all: 42
Value: [0.8861111164093017, 0.75, 1.0, 0.07049209348016061, 0.15811388194561005, 0.0]
Params: 
batch_size: 5
lr: 0.009943353539129438
data_aug: True
ini: O
e_all: 35
Value: [0.8416666746139526, 0.75, 0.9199999570846558, 0.08766518219737346, 0.0, 0.15999998152256012]
Params: 
batch_size: 16
lr: 0.01972804640945201
data_aug: True
ini: G
e_all: 41
Value: [0.8888888955116272, 0.800000011920929, 0.949999988079071, 0.07027283270404318, 0.10000000149011612, 0.10000000149011612]
Params: 
batch_size: 16
lr: 0

In [13]:
seeds = random.sample(range(1,10001), 100)
seeds_print = copy.copy(seeds)

seeds_tr = [random.sample(range(1,10001), 5) for _ in range(100)]
seeds_tr_print = copy.copy(seeds_tr)


##para las semillas de train test split, hago una lista de listas con tantas listas como ejecuciones y cada lista coneteniendo k numeros, según los folds que se hagan
def save_study(study, study_name="study"):
    # Crear una carpeta para guardar el estudio si no existe
    study_dir = os.path.join(os.getcwd(), study_name)
    if not os.path.exists(study_dir):
        os.makedirs(study_dir)
    
    # Guardar los resultados del estudio en un archivo CSV
    study_trials = study.trials_dataframe()
    study_trials.to_csv(os.path.join(study_dir, "study_results.csv"), index=False)

def objective(trial):
    # Define los hiperparámetros para optimizar
    # num_epochs_freeze = trial.suggest_categorical('num_epochs_freeze', [1,3,8,10,15])
    # num_epochs_all = trial.suggest_categorical('num_epochs_all', [25,30,35,40,45,50])
    ##todos estos bs salieron en los mejores, así como ambos extremos del lr
    batch_size = trial.suggest_categorical('batch_size', [5,32, 16])
    lr = trial.suggest_float('lr', 1e-5, 1e-4)
    data_aug = trial.suggest_categorical('data_aug', [True,False])
    ini = trial.suggest_categorical('ini', ['G', 'H', 'O', 'None'])
    all = trial.suggest_int('e_all', 35, 50)
    
    # optimizer_name = trial.suggest_categorical('optimizer_name', ['RMSprop', 'Adam', 'SGD'])

    # # Convertir el nombre del optimizador en una clase PyTorch
    # if optimizer_name == 'Adam':
    #     optimizer_class = torch.optim.Adam
    # elif optimizer_name == 'SGD':
    #     optimizer_class = torch.optim.SGD
    # elif optimizer_name == 'RMSprop':
    #     optimizer_class = torch.optim.RMSprop

    # Ejecuta la función de entrenamiento con los hiperparámetros seleccionados
    # avg_accuracy, avg_recall, avg_precision, acc_std, rec_std, prec_std= prueba_k_fold_resnet_pre(df_mosaic, 5, freeze, all, optim.Adam, batch_size, lr, data_aug, ini, seeds[0], seeds_tr[0])
    avg_accuracy, avg_recall, avg_precision, acc_std, rec_std, prec_std= prueba_k_fold_resnet_no_pre(df_mosaic, 5, all, optim.RMSprop, batch_size, lr, data_aug, ini, seeds[0], seeds_tr[0])
    seeds.pop(0)
    seeds_tr.pop(0)

    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_pre(df_mean_a_2, 5, 5, 35, optim.Adam, batch_size, lr, data_aug, None)
    # avg_l_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre(df_mosaic, 5, all, optim.Adam, batch_size, lr, data_aug, ini, seeds[0], seeds_tr[0])
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre_oc(df_mean_a_2, 5, 40, optim.Adam, batch_size, lr, data_aug, None)

    # Devuelve el valor objetivo (en este caso, la pérdida promedio de validación)
    return avg_accuracy, avg_recall, avg_precision, acc_std, rec_std, prec_std

import optuna
# Supongamos que tienes un dataset llamado 'dataset'
# Realiza una llamada a Optuna para optimizar los hiperparámetros
study = optuna.create_study(directions = ['maximize','maximize','maximize', 'minimize','minimize','minimize'], sampler=None)#None, NSGA por defecto para multiobjetivo, sampler=optuna.samplers.GridSampler(search_space)
print(f"Sampler is {study.sampler.__class__.__name__}")
study.optimize(objective, n_trials=100)#, n_trials=100

# Muestra los mejores hiperparámetros encontrados
print('Best trial:')
trial = study.best_trials
for t in trial:
    print('Value: {}'.format(t.values))
    print('Params: ')
    for key, value in t.params.items():
        print('{}: {}'.format(key, value))

save_study(study, 'estudio_rms')


###hacer tambien lista para las semillas de los train test split en base al numero de folds que haya

[I 2024-04-26 18:25:21,931] A new study created in memory with name: no-name-acdb9274-e088-4611-95bc-eacb25048cdc


Sampler is NSGAIISampler
Fold 1
Best epoc: 49. Best val loss: 0.11343538761138916
Accuracy: 0.8888888955116272. Recall: 1.0. Precision: 0.800000011920929
Fold 2
Best epoc: 18. Best val loss: 0.2654600739479065
Accuracy: 0.7777777910232544. Recall: 0.5. Precision: 1.0
Fold 3
Best epoc: 35. Best val loss: 0.035214316099882126
Accuracy: 0.7777777910232544. Recall: 0.75. Precision: 0.75
Fold 4
Best epoc: 49. Best val loss: 0.14169105887413025
Accuracy: 0.6666666865348816. Recall: 0.25. Precision: 1.0
Fold 5


[I 2024-04-26 18:26:11,158] Trial 0 finished with values: [0.8222222328186035, 0.699999988079071, 0.9100000262260437, 0.11331153799261195, 0.29154759645462036, 0.11135528236627579] and parameters: {'batch_size': 32, 'lr': 5.592661420015295e-05, 'data_aug': False, 'ini': 'None', 'e_all': 50}. 


Best epoc: 23. Best val loss: 0.31176191568374634
Accuracy: 1.0. Recall: 1.0. Precision: 1.0
Fold 1
Best epoc: 12. Best val loss: 0.008687110850587487
Accuracy: 0.8888888955116272. Recall: 1.0. Precision: 0.800000011920929
Fold 2
Best epoc: 2. Best val loss: 0.004165889404248446
Accuracy: 0.7777777910232544. Recall: 0.5. Precision: 1.0
Fold 3
Best epoc: 27. Best val loss: 0.4564882591366768
Accuracy: 0.7777777910232544. Recall: 1.0. Precision: 0.6666666865348816
Fold 4
Best epoc: 19. Best val loss: 0.15435554087162018
Accuracy: 0.8888888955116272. Recall: 0.75. Precision: 1.0
Fold 5


[I 2024-04-26 18:26:56,458] Trial 1 finished with values: [0.8416666746139526, 0.800000011920929, 0.8933333158493042, 0.052411002217552065, 0.18708285689353943, 0.13727505505084991] and parameters: {'batch_size': 5, 'lr': 9.014452751089779e-05, 'data_aug': True, 'ini': 'O', 'e_all': 44}. 


Best epoc: 25. Best val loss: 0.18167046457529068
Accuracy: 0.875. Recall: 0.75. Precision: 1.0
Fold 1
Best epoc: 14. Best val loss: 0.5694021582603455
Accuracy: 0.8888888955116272. Recall: 0.75. Precision: 1.0
Fold 2
Best epoc: 35. Best val loss: 0.33292222023010254
Accuracy: 0.7777777910232544. Recall: 0.75. Precision: 0.75
Fold 3
Best epoc: 30. Best val loss: 0.5672549605369568
Accuracy: 0.8888888955116272. Recall: 1.0. Precision: 0.800000011920929
Fold 4
Best epoc: 46. Best val loss: 0.08867830783128738
Accuracy: 0.7777777910232544. Recall: 0.75. Precision: 0.75
Fold 5


... (192 outputs left out)

[I 2024-04-26 19:36:56,906] Trial 98 finished with values: [0.8444444537162781, 0.800000011920929, 0.8700000047683716, 0.11331153799261195, 0.29154759645462036, 0.10770329087972641] and parameters: {'batch_size': 32, 'lr': 8.864504818046978e-05, 'data_aug': True, 'ini': 'H', 'e_all': 38}. 


Best epoc: 37. Best val loss: 0.0023147305473685265
Accuracy: 1.0. Recall: 1.0. Precision: 1.0
Fold 1
Best epoc: 9. Best val loss: 0.5555164217948914
Accuracy: 0.6666666865348816. Recall: 0.75. Precision: 0.6000000238418579
Fold 2
Best epoc: 4. Best val loss: 0.10580369085073471
Accuracy: 1.0. Recall: 1.0. Precision: 1.0
Fold 3
Best epoc: 18. Best val loss: 0.03743559122085571
Accuracy: 0.8888888955116272. Recall: 0.75. Precision: 1.0
Fold 4
Best epoc: 4. Best val loss: 0.5375786423683167
Accuracy: 0.5555555820465088. Recall: 0.75. Precision: 0.5
Fold 5


[I 2024-04-26 19:37:40,648] Trial 99 finished with values: [0.8222222328186035, 0.8500000238418579, 0.8199999928474426, 0.18053417600901195, 0.12247449159622192, 0.22271056473255157] and parameters: {'batch_size': 16, 'lr': 6.98494099256818e-05, 'data_aug': True, 'ini': 'H', 'e_all': 50}. 


Best epoc: 10. Best val loss: 0.027695316821336746
Accuracy: 1.0. Recall: 1.0. Precision: 1.0
Best trial:
Value: [0.819444453716278, 0.8999999761581421, 0.7766667008399963, 0.05121968676021562, 0.12247449159622192, 0.12274634838104248]
Params: 
batch_size: 32
lr: 7.312594340744105e-05
data_aug: True
ini: G
e_all: 45
Value: [0.9333333373069763, 0.8500000238418579, 1.0, 0.054433102150715826, 0.12247449159622192, 0.0]
Params: 
batch_size: 5
lr: 5.021770546387372e-05
data_aug: True
ini: G
e_all: 43
Value: [0.9333333373069763, 0.8500000238418579, 1.0, 0.054433102150715826, 0.12247449159622192, 0.0]
Params: 
batch_size: 5
lr: 7.312594340744105e-05
data_aug: True
ini: None
e_all: 49
Value: [0.8416666746139526, 0.8500000238418579, 0.8300000429153442, 0.052411002217552065, 0.20000000298023224, 0.08717797696590424]
Params: 
batch_size: 5
lr: 7.312594340744105e-05
data_aug: False
ini: O
e_all: 49
Value: [0.8416666746139526, 0.75, 0.8999999761581421, 0.052411002217552065, 0.0, 0.12247449159622192]

## PRUEBA SIN TL CON OC

In [11]:
seeds = random.sample(range(1,10001), 100)
seeds_print = copy.copy(seeds)

seeds_tr = [random.sample(range(1,10001), 5) for _ in range(100)]
seeds_tr_print = copy.copy(seeds_tr)


##para las semillas de train test split, hago una lista de listas con tantas listas como ejecuciones y cada lista coneteniendo k numeros, según los folds que se hagan
def save_study(study, study_name="study"):
    # Crear una carpeta para guardar el estudio si no existe
    study_dir = os.path.join(os.getcwd(), study_name)
    if not os.path.exists(study_dir):
        os.makedirs(study_dir)
    
    # Guardar los resultados del estudio en un archivo CSV
    study_trials = study.trials_dataframe()
    study_trials.to_csv(os.path.join(study_dir, "study_results.csv"), index=False)

def objective(trial):
    # Define los hiperparámetros para optimizar
    # num_epochs_freeze = trial.suggest_categorical('num_epochs_freeze', [1,3,8,10,15])
    # num_epochs_all = trial.suggest_categorical('num_epochs_all', [25,30,35,40,45,50])
    ##todos estos bs salieron en los mejores, así como ambos extremos del lr
    batch_size = trial.suggest_categorical('batch_size', [5,32, 8])
    lr = trial.suggest_float('lr', 1e-5, 1e-4)
    data_aug = trial.suggest_categorical('data_aug', [True,False])
    ini = trial.suggest_categorical('ini', ['G', 'H', 'O', 'None'])
    all = trial.suggest_int('e_all', 35, 50)
    
    # optimizer_name = trial.suggest_categorical('optimizer_name', ['RMSprop', 'Adam', 'SGD'])

    # # Convertir el nombre del optimizador en una clase PyTorch
    # if optimizer_name == 'Adam':
    #     optimizer_class = torch.optim.Adam
    # elif optimizer_name == 'SGD':
    #     optimizer_class = torch.optim.SGD
    # elif optimizer_name == 'RMSprop':
    #     optimizer_class = torch.optim.RMSprop

    # Ejecuta la función de entrenamiento con los hiperparámetros seleccionados
    # avg_accuracy, avg_recall, avg_precision, acc_std, rec_std, prec_std= prueba_k_fold_resnet_pre(df_mosaic, 5, freeze, all, optim.Adam, batch_size, lr, data_aug, ini, seeds[0], seeds_tr[0])
    avg_accuracy, avg_recall, avg_precision, acc_std, rec_std, prec_std= prueba_k_fold_resnet_no_pre_oc(df_mosaic, 5, all, optim.Adam, batch_size, lr, data_aug, ini, seeds[0], seeds_tr[0])
    seeds.pop(0)
    seeds_tr.pop(0)

    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_pre(df_mean_a_2, 5, 5, 35, optim.Adam, batch_size, lr, data_aug, None)
    # avg_l_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre(df_mosaic, 5, all, optim.Adam, batch_size, lr, data_aug, ini, seeds[0], seeds_tr[0])
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre_oc(df_mean_a_2, 5, 40, optim.Adam, batch_size, lr, data_aug, None)

    # Devuelve el valor objetivo (en este caso, la pérdida promedio de validación)
    return avg_accuracy, avg_recall, avg_precision, acc_std, rec_std, prec_std

import optuna
# Supongamos que tienes un dataset llamado 'dataset'
# Realiza una llamada a Optuna para optimizar los hiperparámetros
study = optuna.create_study(directions = ['maximize','maximize','maximize', 'minimize','minimize','minimize'], sampler=None)#None, NSGA por defecto para multiobjetivo, sampler=optuna.samplers.GridSampler(search_space)
print(f"Sampler is {study.sampler.__class__.__name__}")
study.optimize(objective, n_trials=100)#, n_trials=100

# Muestra los mejores hiperparámetros encontrados
print('Best trial:')
trial = study.best_trials
for t in trial:
    print('Value: {}'.format(t.values))
    print('Params: ')
    for key, value in t.params.items():
        print('{}: {}'.format(key, value))

save_study(study, 'estudio_oc_adam')


###hacer tambien lista para las semillas de los train test split en base al numero de folds que haya

[I 2024-04-26 16:01:38,775] A new study created in memory with name: no-name-a7ac9353-378e-4ce4-bb18-a48444698493


Sampler is NSGAIISampler
Fold 1
Best epoc: 22. Best val loss: 0.32500001788139343
Accuracy: 0.6666666865348816. Recall: 0.5. Precision: 0.6666666865348816
Fold 2
Best epoc: 41. Best val loss: 0.21918269991874695
Accuracy: 0.8888888955116272. Recall: 0.75. Precision: 1.0
Fold 3
Best epoc: 20. Best val loss: 0.19348222017288208
Accuracy: 1.0. Recall: 1.0. Precision: 1.0
Fold 4
Best epoc: 1. Best val loss: 0.7552889585494995
Accuracy: 0.3333333432674408. Recall: 0.5. Precision: 0.3333333432674408
Fold 5


[I 2024-04-26 16:02:22,995] Trial 0 finished with values: [0.7527777850627899, 0.699999988079071, 0.800000011920929, 0.23576771993338205, 0.18708285689353943, 0.2666666507720947] and parameters: {'batch_size': 8, 'lr': 3.416941357845722e-05, 'data_aug': True, 'ini': 'H', 'e_all': 45}. 


Best epoc: 36. Best val loss: 0.24181458353996277
Accuracy: 0.875. Recall: 0.75. Precision: 1.0
Fold 1
Best epoc: 0. Best val loss: 0.7492464780807495
Accuracy: 0.5555555820465088. Recall: 0.0. Precision: 0.0
Fold 2
Best epoc: 49. Best val loss: 0.5493441224098206
Accuracy: 1.0. Recall: 1.0. Precision: 1.0
Fold 3
Best epoc: 46. Best val loss: 0.33071741461753845
Accuracy: 0.8888888955116272. Recall: 0.75. Precision: 1.0
Fold 4
Best epoc: 25. Best val loss: 0.6558021306991577
Accuracy: 0.8888888955116272. Recall: 0.75. Precision: 1.0
Fold 5


[I 2024-04-26 16:03:13,179] Trial 1 finished with values: [0.8166666746139526, 0.6499999761581421, 0.75, 0.15275251536111487, 0.33911648392677307, 0.3872983346207417] and parameters: {'batch_size': 32, 'lr': 4.222625481764003e-05, 'data_aug': False, 'ini': 'O', 'e_all': 50}. 


Best epoc: 39. Best val loss: 0.11369441449642181
Accuracy: 0.75. Recall: 0.75. Precision: 0.75
Fold 1
Best epoc: 28. Best val loss: 0.08716001361608505
Accuracy: 0.7777777910232544. Recall: 0.5. Precision: 1.0
Fold 2
Best epoc: 40. Best val loss: 0.10375598818063736
Accuracy: 0.7777777910232544. Recall: 0.5. Precision: 1.0
Fold 3
Best epoc: 20. Best val loss: 0.5643796324729919
Accuracy: 0.8888888955116272. Recall: 1.0. Precision: 0.800000011920929
Fold 4
Best epoc: 8. Best val loss: 0.5519129633903503
Accuracy: 0.7777777910232544. Recall: 0.5. Precision: 1.0
Fold 5


... (192 outputs left out)

[I 2024-04-26 17:17:33,894] Trial 98 finished with values: [0.7972222328186035, 0.8500000238418579, 0.7866666913032532, 0.0803157287408968, 0.12247449159622192, 0.17587871849536896] and parameters: {'batch_size': 5, 'lr': 8.826867454524155e-05, 'data_aug': True, 'ini': 'None', 'e_all': 38}. 


Best epoc: 13. Best val loss: 0.012023441027849913
Accuracy: 0.875. Recall: 0.75. Precision: 1.0
Fold 1
Best epoc: 18. Best val loss: 0.34378771483898163
Accuracy: 0.8888888955116272. Recall: 1.0. Precision: 0.800000011920929
Fold 2
Best epoc: 24. Best val loss: 0.11709542199969292
Accuracy: 0.7777777910232544. Recall: 0.5. Precision: 1.0
Fold 3
Best epoc: 28. Best val loss: 0.05977033171802759
Accuracy: 0.8888888955116272. Recall: 0.75. Precision: 1.0
Fold 4
Best epoc: 21. Best val loss: 0.0742704588919878
Accuracy: 0.6666666865348816. Recall: 0.75. Precision: 0.6000000238418579
Fold 5


[I 2024-04-26 17:18:18,488] Trial 99 finished with values: [0.819444453716278, 0.800000011920929, 0.8400000333786011, 0.08695819298987903, 0.18708285689353943, 0.14966627955436707] and parameters: {'batch_size': 5, 'lr': 7.443966677904427e-05, 'data_aug': True, 'ini': 'O', 'e_all': 39}. 


Best epoc: 35. Best val loss: 0.030415184795856476
Accuracy: 0.875. Recall: 1.0. Precision: 0.800000011920929
Best trial:
Value: [0.8861111164093017, 0.75, 1.0, 0.07049209348016063, 0.15811388194561005, 0.0]
Params: 
batch_size: 5
lr: 9.83023066015208e-05
data_aug: True
ini: O
e_all: 49
Value: [0.8861111164093017, 0.800000011920929, 0.9600000381469727, 0.005555558204650879, 0.10000000149011612, 0.07999999821186066]
Params: 
batch_size: 8
lr: 2.1906968027735773e-05
data_aug: True
ini: G
e_all: 49
Value: [0.8861111164093017, 0.8500000238418579, 0.9100000262260437, 0.07049209348016063, 0.12247449159622192, 0.11135528236627579]
Params: 
batch_size: 5
lr: 2.7910998603808262e-05
data_aug: True
ini: G
e_all: 38
Value: [0.9083333373069763, 0.8500000238418579, 0.9600000381469727, 0.08407080641548241, 0.20000000298023224, 0.07999999821186066]
Params: 
batch_size: 8
lr: 6.999718450799375e-05
data_aug: True
ini: H
e_all: 50
Value: [0.9111111164093018, 0.800000011920929, 1.0, 0.08314793697230627, 0

In [12]:
seeds = random.sample(range(1,10001), 100)
seeds_print = copy.copy(seeds)

seeds_tr = [random.sample(range(1,10001), 5) for _ in range(100)]
seeds_tr_print = copy.copy(seeds_tr)


##para las semillas de train test split, hago una lista de listas con tantas listas como ejecuciones y cada lista coneteniendo k numeros, según los folds que se hagan
def save_study(study, study_name="study"):
    # Crear una carpeta para guardar el estudio si no existe
    study_dir = os.path.join(os.getcwd(), study_name)
    if not os.path.exists(study_dir):
        os.makedirs(study_dir)
    
    # Guardar los resultados del estudio en un archivo CSV
    study_trials = study.trials_dataframe()
    study_trials.to_csv(os.path.join(study_dir, "study_results.csv"), index=False)

def objective(trial):
    # Define los hiperparámetros para optimizar
    # num_epochs_freeze = trial.suggest_categorical('num_epochs_freeze', [1,3,8,10,15])
    # num_epochs_all = trial.suggest_categorical('num_epochs_all', [25,30,35,40,45,50])
    ##todos estos bs salieron en los mejores, así como ambos extremos del lr
    batch_size = trial.suggest_categorical('batch_size', [5,16, 8])
    lr = trial.suggest_float('lr', 0.001, 0.01)
    data_aug = trial.suggest_categorical('data_aug', [True,False])
    ini = trial.suggest_categorical('ini', ['G', 'H', 'O', 'None'])
    all = trial.suggest_int('e_all', 35, 50)
    
    # optimizer_name = trial.suggest_categorical('optimizer_name', ['RMSprop', 'Adam', 'SGD'])

    # # Convertir el nombre del optimizador en una clase PyTorch
    # if optimizer_name == 'Adam':
    #     optimizer_class = torch.optim.Adam
    # elif optimizer_name == 'SGD':
    #     optimizer_class = torch.optim.SGD
    # elif optimizer_name == 'RMSprop':
    #     optimizer_class = torch.optim.RMSprop

    # Ejecuta la función de entrenamiento con los hiperparámetros seleccionados
    # avg_accuracy, avg_recall, avg_precision, acc_std, rec_std, prec_std= prueba_k_fold_resnet_pre(df_mosaic, 5, freeze, all, optim.Adam, batch_size, lr, data_aug, ini, seeds[0], seeds_tr[0])
    avg_accuracy, avg_recall, avg_precision, acc_std, rec_std, prec_std= prueba_k_fold_resnet_no_pre_oc(df_mosaic, 5, all, optim.SGD, batch_size, lr, data_aug, ini, seeds[0], seeds_tr[0])
    seeds.pop(0)
    seeds_tr.pop(0)

    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_pre(df_mean_a_2, 5, 5, 35, optim.Adam, batch_size, lr, data_aug, None)
    # avg_l_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre(df_mosaic, 5, all, optim.Adam, batch_size, lr, data_aug, ini, seeds[0], seeds_tr[0])
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre_oc(df_mean_a_2, 5, 40, optim.Adam, batch_size, lr, data_aug, None)

    # Devuelve el valor objetivo (en este caso, la pérdida promedio de validación)
    return avg_accuracy, avg_recall, avg_precision, acc_std, rec_std, prec_std

import optuna
# Supongamos que tienes un dataset llamado 'dataset'
# Realiza una llamada a Optuna para optimizar los hiperparámetros
study = optuna.create_study(directions = ['maximize','maximize','maximize', 'minimize','minimize','minimize'], sampler=None)#None, NSGA por defecto para multiobjetivo, sampler=optuna.samplers.GridSampler(search_space)
print(f"Sampler is {study.sampler.__class__.__name__}")
study.optimize(objective, n_trials=100)#, n_trials=100

# Muestra los mejores hiperparámetros encontrados
print('Best trial:')
trial = study.best_trials
for t in trial:
    print('Value: {}'.format(t.values))
    print('Params: ')
    for key, value in t.params.items():
        print('{}: {}'.format(key, value))

save_study(study, 'estudio_oc_sgd')


###hacer tambien lista para las semillas de los train test split en base al numero de folds que haya

[I 2024-04-26 17:18:18,538] A new study created in memory with name: no-name-711f1481-adf7-4c3e-aa96-eb2017e51e38


Sampler is NSGAIISampler
Fold 1
Best epoc: 24. Best val loss: 0.010157706215977669
Accuracy: 0.8888888955116272. Recall: 0.75. Precision: 1.0
Fold 2
Best epoc: 7. Best val loss: 0.08940877765417099
Accuracy: 0.8888888955116272. Recall: 0.75. Precision: 1.0
Fold 3
Best epoc: 19. Best val loss: 0.45970919728279114
Accuracy: 0.8888888955116272. Recall: 0.75. Precision: 1.0
Fold 4
Best epoc: 24. Best val loss: 0.011334609240293503
Accuracy: 0.6666666865348816. Recall: 0.25. Precision: 1.0
Fold 5


[I 2024-04-26 17:18:55,920] Trial 0 finished with values: [0.8166666746139526, 0.6499999761581421, 0.949999988079071, 0.09229581735049708, 0.20000000298023224, 0.10000000149011612] and parameters: {'batch_size': 8, 'lr': 0.0063175781856142105, 'data_aug': False, 'ini': 'None', 'e_all': 39}. 


Best epoc: 18. Best val loss: 0.11698820441961288
Accuracy: 0.75. Recall: 0.75. Precision: 0.75
Fold 1
Best epoc: 34. Best val loss: 0.024392610415816307
Accuracy: 0.8888888955116272. Recall: 0.75. Precision: 1.0
Fold 2
Best epoc: 2. Best val loss: 0.4960813820362091
Accuracy: 0.7777777910232544. Recall: 0.5. Precision: 1.0
Fold 3
Best epoc: 11. Best val loss: 0.11220894753932953
Accuracy: 0.7777777910232544. Recall: 1.0. Precision: 0.6666666865348816
Fold 4
Best epoc: 20. Best val loss: 0.004764804616570473
Accuracy: 0.8888888955116272. Recall: 0.75. Precision: 1.0
Fold 5


[I 2024-04-26 17:19:35,696] Trial 1 finished with values: [0.7916666746139527, 0.6499999761581421, 0.9333333969116211, 0.09702360854371224, 0.25495097041130066, 0.13333332538604736] and parameters: {'batch_size': 8, 'lr': 0.006428665201730465, 'data_aug': False, 'ini': 'G', 'e_all': 42}. 


Best epoc: 37. Best val loss: 0.0032505644485354424
Accuracy: 0.625. Recall: 0.25. Precision: 1.0
Fold 1
Best epoc: 20. Best val loss: 3.0640977229268174e-05
Accuracy: 0.7777777910232544. Recall: 0.5. Precision: 1.0
Fold 2
Best epoc: 38. Best val loss: 0.0028623833204619586
Accuracy: 0.7777777910232544. Recall: 0.5. Precision: 1.0
Fold 3
Best epoc: 4. Best val loss: 0.18251939862966537
Accuracy: 0.7777777910232544. Recall: 0.75. Precision: 0.75
Fold 4
Best epoc: 4. Best val loss: 0.5157994329929352
Accuracy: 0.6666666865348816. Recall: 1.0. Precision: 0.5714285969734192
Fold 5


... (192 outputs left out)

[I 2024-04-26 18:24:43,124] Trial 98 finished with values: [0.8638888955116272, 0.8500000238418579, 0.8933333158493042, 0.12921891827663284, 0.29999998211860657, 0.13727505505084991] and parameters: {'batch_size': 8, 'lr': 0.0063175781856142105, 'data_aug': False, 'ini': 'None', 'e_all': 35}. 


Best epoc: 5. Best val loss: 0.3122551441192627
Accuracy: 0.875. Recall: 1.0. Precision: 0.800000011920929
Fold 1
Best epoc: 7. Best val loss: 0.14372864365577698
Accuracy: 0.7777777910232544. Recall: 0.5. Precision: 1.0
Fold 2
Best epoc: 18. Best val loss: 8.524560689693317e-05
Accuracy: 0.8888888955116272. Recall: 0.75. Precision: 1.0
Fold 3
Best epoc: 20. Best val loss: 5.6596989452373236e-05
Accuracy: 1.0. Recall: 1.0. Precision: 1.0
Fold 4
Best epoc: 7. Best val loss: 0.2316516637802124
Accuracy: 0.6666666865348816. Recall: 0.75. Precision: 0.6000000238418579
Fold 5


[I 2024-04-26 18:25:21,842] Trial 99 finished with values: [0.7916666746139527, 0.75, 0.8399999737739563, 0.13888888597488422, 0.15811388194561005, 0.1959591805934906] and parameters: {'batch_size': 8, 'lr': 0.0074082511732717436, 'data_aug': False, 'ini': 'G', 'e_all': 41}. 


Best epoc: 25. Best val loss: 0.053193364292383194
Accuracy: 0.625. Recall: 0.75. Precision: 0.6000000238418579
Best trial:
Value: [0.8416666746139526, 0.75, 0.9333333969116211, 0.052411002217552065, 0.15811388194561005, 0.13333332538604736]
Params: 
batch_size: 5
lr: 0.004722011491107303
data_aug: False
ini: O
e_all: 36
Value: [0.8638888955116272, 0.699999988079071, 1.0, 0.08258926569525993, 0.18708285689353943, 0.0]
Params: 
batch_size: 5
lr: 0.009227610753477396
data_aug: False
ini: None
e_all: 37
Value: [0.8611111164093017, 0.8500000238418579, 0.8799999952316284, 0.05555555820465088, 0.20000000298023224, 0.0979795902967453]
Params: 
batch_size: 5
lr: 0.008081337952860168
data_aug: False
ini: H
e_all: 45
Value: [0.7750000119209289, 0.75, 0.8199999928474426, 0.1384437217204118, 0.0, 0.22271056473255157]
Params: 
batch_size: 5
lr: 0.0036265252270777045
data_aug: True
ini: H
e_all: 45
Value: [0.7500000119209289, 0.75, 0.7533333897590637, 0.04303314572621842, 0.15811388194561005, 0.1355

In [16]:
seeds = random.sample(range(1,10001), 100)
seeds_print = copy.copy(seeds)

seeds_tr = [random.sample(range(1,10001), 5) for _ in range(100)]
seeds_tr_print = copy.copy(seeds_tr)


##para las semillas de train test split, hago una lista de listas con tantas listas como ejecuciones y cada lista coneteniendo k numeros, según los folds que se hagan
def save_study(study, study_name="study"):
    # Crear una carpeta para guardar el estudio si no existe
    study_dir = os.path.join(os.getcwd(), study_name)
    if not os.path.exists(study_dir):
        os.makedirs(study_dir)
    
    # Guardar los resultados del estudio en un archivo CSV
    study_trials = study.trials_dataframe()
    study_trials.to_csv(os.path.join(study_dir, "study_results.csv"), index=False)

def objective(trial):
    # Define los hiperparámetros para optimizar
    # num_epochs_freeze = trial.suggest_categorical('num_epochs_freeze', [1,3,8,10,15])
    # num_epochs_all = trial.suggest_categorical('num_epochs_all', [25,30,35,40,45,50])
    ##todos estos bs salieron en los mejores, así como ambos extremos del lr
    batch_size = trial.suggest_categorical('batch_size', [5,32])
    lr = trial.suggest_float('lr', 1e-5/2, 1e-5*2)
    data_aug = trial.suggest_categorical('data_aug', [True,False])
    ini = trial.suggest_categorical('ini', ['G', 'H', 'O', 'None'])
    all = trial.suggest_int('e_all', 35, 50)
    
    # optimizer_name = trial.suggest_categorical('optimizer_name', ['RMSprop', 'Adam', 'SGD'])

    # # Convertir el nombre del optimizador en una clase PyTorch
    # if optimizer_name == 'Adam':
    #     optimizer_class = torch.optim.Adam
    # elif optimizer_name == 'SGD':
    #     optimizer_class = torch.optim.SGD
    # elif optimizer_name == 'RMSprop':
    #     optimizer_class = torch.optim.RMSprop

    # Ejecuta la función de entrenamiento con los hiperparámetros seleccionados
    # avg_accuracy, avg_recall, avg_precision, acc_std, rec_std, prec_std= prueba_k_fold_resnet_pre(df_mosaic, 5, freeze, all, optim.Adam, batch_size, lr, data_aug, ini, seeds[0], seeds_tr[0])
    avg_accuracy, avg_recall, avg_precision, acc_std, rec_std, prec_std= prueba_k_fold_resnet_no_pre_oc(df_mosaic, 5, all, optim.RMSprop, batch_size, lr, data_aug, ini, seeds[0], seeds_tr[0])
    seeds.pop(0)
    seeds_tr.pop(0)

    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_pre(df_mean_a_2, 5, 5, 35, optim.Adam, batch_size, lr, data_aug, None)
    # avg_l_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre(df_mosaic, 5, all, optim.Adam, batch_size, lr, data_aug, ini, seeds[0], seeds_tr[0])
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre_oc(df_mean_a_2, 5, 40, optim.Adam, batch_size, lr, data_aug, None)

    # Devuelve el valor objetivo (en este caso, la pérdida promedio de validación)
    return avg_accuracy, avg_recall, avg_precision, acc_std, rec_std, prec_std

import optuna
# Supongamos que tienes un dataset llamado 'dataset'
# Realiza una llamada a Optuna para optimizar los hiperparámetros
study = optuna.create_study(directions = ['maximize','maximize','maximize', 'minimize','minimize','minimize'], sampler=None)#None, NSGA por defecto para multiobjetivo, sampler=optuna.samplers.GridSampler(search_space)
print(f"Sampler is {study.sampler.__class__.__name__}")
study.optimize(objective, n_trials=100)#, n_trials=100

# Muestra los mejores hiperparámetros encontrados
print('Best trial:')
trial = study.best_trials
for t in trial:
    print('Value: {}'.format(t.values))
    print('Params: ')
    for key, value in t.params.items():
        print('{}: {}'.format(key, value))

save_study(study, 'estudio_oc_rms')


###hacer tambien lista para las semillas de los train test split en base al numero de folds que haya

[I 2024-04-26 21:59:21,774] A new study created in memory with name: no-name-edccabff-31f8-45e4-ac77-89b14ddba28f


Sampler is NSGAIISampler
Fold 1
Best epoc: 0. Best val loss: 0.7154815196990967
Accuracy: 0.4444444477558136. Recall: 1.0. Precision: 0.4444444477558136
Fold 2
Best epoc: 19. Best val loss: 2.0776394649146823e-06
Accuracy: 0.6666666865348816. Recall: 0.25. Precision: 1.0
Fold 3
Best epoc: 18. Best val loss: 0.0010066470131278038
Accuracy: 1.0. Recall: 1.0. Precision: 1.0
Fold 4
Best epoc: 37. Best val loss: 0.4495077431201935
Accuracy: 1.0. Recall: 1.0. Precision: 1.0
Fold 5


[I 2024-04-26 22:00:07,404] Trial 0 finished with values: [0.747222226858139, 0.8500000238418579, 0.803174614906311, 0.21949717835188096, 0.29999998211860657, 0.2443825900554657] and parameters: {'batch_size': 32, 'lr': 1.5959370277272823e-05, 'data_aug': True, 'ini': 'G', 'e_all': 45}. 


Best epoc: 13. Best val loss: 0.5584444999694824
Accuracy: 0.625. Recall: 1.0. Precision: 0.5714285969734192
Fold 1
Best epoc: 14. Best val loss: 0.21970784664154053
Accuracy: 0.8888888955116272. Recall: 0.75. Precision: 1.0
Fold 2
Best epoc: 39. Best val loss: 0.5523200035095215
Accuracy: 0.7777777910232544. Recall: 0.5. Precision: 1.0
Fold 3
Best epoc: 39. Best val loss: 0.2702758312225342
Accuracy: 0.7777777910232544. Recall: 1.0. Precision: 0.6666666865348816
Fold 4
Best epoc: 18. Best val loss: 0.3542632460594177
Accuracy: 0.7777777910232544. Recall: 0.5. Precision: 1.0
Fold 5


[I 2024-04-26 22:00:49,203] Trial 1 finished with values: [0.819444453716278, 0.699999988079071, 0.9333333969116211, 0.05121968676021561, 0.18708285689353943, 0.13333332538604736] and parameters: {'batch_size': 32, 'lr': 6.42435151745096e-06, 'data_aug': False, 'ini': 'O', 'e_all': 40}. 


Best epoc: 14. Best val loss: 0.06288543343544006
Accuracy: 0.875. Recall: 0.75. Precision: 1.0
Fold 1
Best epoc: 10. Best val loss: 0.3467156982515007
Accuracy: 0.8888888955116272. Recall: 1.0. Precision: 0.800000011920929
Fold 2
Best epoc: 20. Best val loss: 0.2057071477174759
Accuracy: 0.8888888955116272. Recall: 0.75. Precision: 1.0
Fold 3
Best epoc: 28. Best val loss: 0.042896334547549486
Accuracy: 0.8888888955116272. Recall: 0.75. Precision: 1.0
Fold 4
Best epoc: 5. Best val loss: 0.028619012620765716
Accuracy: 0.8888888955116272. Recall: 0.75. Precision: 1.0
Fold 5


... (192 outputs left out)

[I 2024-04-26 23:13:19,313] Trial 98 finished with values: [0.7083333492279053, 0.3499999940395355, 0.8, 0.10900786272626065, 0.25495100021362305, 0.4000000000000001] and parameters: {'batch_size': 32, 'lr': 1.969118200527281e-05, 'data_aug': False, 'ini': 'G', 'e_all': 47}. 


Best epoc: 19. Best val loss: 0.017311127856373787
Accuracy: 0.875. Recall: 0.75. Precision: 1.0
Fold 1
Best epoc: 31. Best val loss: 0.014367741532623768
Accuracy: 0.8888888955116272. Recall: 0.75. Precision: 1.0
Fold 2
Best epoc: 4. Best val loss: 0.6649113893508911
Accuracy: 0.5555555820465088. Recall: 0.0. Precision: 0.0
Fold 3
Best epoc: 25. Best val loss: 0.39813464879989624
Accuracy: 0.8888888955116272. Recall: 1.0. Precision: 0.800000011920929
Fold 4
Best epoc: 45. Best val loss: 0.05692020058631897
Accuracy: 0.7777777910232544. Recall: 0.5. Precision: 1.0
Fold 5


[I 2024-04-26 23:14:04,506] Trial 99 finished with values: [0.8222222328186035, 0.6499999761581421, 0.7600000023841857, 0.15071843508593108, 0.37416571378707886, 0.3878143888392164] and parameters: {'batch_size': 32, 'lr': 5.519521068619502e-06, 'data_aug': True, 'ini': 'None', 'e_all': 46}. 


Best epoc: 45. Best val loss: 0.08780442178249359
Accuracy: 1.0. Recall: 1.0. Precision: 1.0
Best trial:
Value: [0.8861111164093017, 0.800000011920929, 0.9600000381469727, 0.005555558204650879, 0.10000000149011612, 0.07999999821186066]
Params: 
batch_size: 5
lr: 1.5185733901892692e-05
data_aug: True
ini: H
e_all: 39
Value: [0.8888888955116272, 0.800000011920929, 0.949999988079071, 0.09938079307643344, 0.18708285689353943, 0.10000000149011612]
Params: 
batch_size: 5
lr: 1.4794971236111908e-05
data_aug: False
ini: H
e_all: 50
Value: [0.8888888955116272, 0.800000011920929, 0.949999988079071, 0.09938079307643344, 0.18708285689353943, 0.10000000149011612]
Params: 
batch_size: 5
lr: 1.2871335131480058e-05
data_aug: True
ini: O
e_all: 49
Value: [0.9055555582046508, 0.8500000238418579, 0.9600000381469727, 0.09229582022071937, 0.20000000298023224, 0.07999999821186066]
Params: 
batch_size: 32
lr: 1.1649075024444795e-05
data_aug: True
ini: H
e_all: 40
Value: [0.8638888955116272, 0.699999988079071

## PRUEBA CON TL SIN OC, HACIENDO 75 PRUEBAS

In [5]:
seeds = random.sample(range(1,10001), 100)
seeds_print = copy.copy(seeds)

seeds_tr = [random.sample(range(1,10001), 5) for _ in range(100)]
seeds_tr_print = copy.copy(seeds_tr)


##para las semillas de train test split, hago una lista de listas con tantas listas como ejecuciones y cada lista coneteniendo k numeros, según los folds que se hagan
def save_study(study, study_name="study"):
    # Crear una carpeta para guardar el estudio si no existe
    study_dir = os.path.join(os.getcwd(), study_name)
    if not os.path.exists(study_dir):
        os.makedirs(study_dir)
    
    # Guardar los resultados del estudio en un archivo CSV
    study_trials = study.trials_dataframe()
    study_trials.to_csv(os.path.join(study_dir, "study_results.csv"), index=False)

def objective(trial):
    # Define los hiperparámetros para optimizar
    # num_epochs_freeze = trial.suggest_categorical('num_epochs_freeze', [1,3,8,10,15])
    # num_epochs_all = trial.suggest_categorical('num_epochs_all', [25,30,35,40,45,50])
    ##todos estos bs salieron en los mejores, así como ambos extremos del lr
    batch_size = trial.suggest_categorical('batch_size', [16,5,32])
    lr = trial.suggest_float('lr', 1e-5, 1e-4)
    data_aug = trial.suggest_categorical('data_aug', [True,False])
    ini = trial.suggest_categorical('ini', ['G', 'H', 'O', 'None'])
    freeze = trial.suggest_int('e_freeze', 3, 7)
    all = trial.suggest_int('e_all', 35, 50)
    
    # optimizer_name = trial.suggest_categorical('optimizer_name', ['RMSprop', 'Adam', 'SGD'])

    # # Convertir el nombre del optimizador en una clase PyTorch
    # if optimizer_name == 'Adam':
    #     optimizer_class = torch.optim.Adam
    # elif optimizer_name == 'SGD':
    #     optimizer_class = torch.optim.SGD
    # elif optimizer_name == 'RMSprop':
    #     optimizer_class = torch.optim.RMSprop

    # Ejecuta la función de entrenamiento con los hiperparámetros seleccionados
    avg_accuracy, avg_recall, avg_precision, acc_std, rec_std, prec_std= prueba_k_fold_resnet_pre(df_mosaic, 5, freeze, all, optim.Adam, batch_size, lr, data_aug, ini, seeds[0], seeds_tr[0])
    seeds.pop(0)
    seeds_tr.pop(0)

    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_pre(df_mean_a_2, 5, 5, 35, optim.Adam, batch_size, lr, data_aug, None)
    # avg_l_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre(df_mean_a_2, 5, 40, optim.Adam, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre_oc(df_mean_a_2, 5, 40, optim.Adam, batch_size, lr, data_aug, None)

    # Devuelve el valor objetivo (en este caso, la pérdida promedio de validación)
    return avg_accuracy, avg_recall, avg_precision, acc_std, rec_std, prec_std

import optuna
# Supongamos que tienes un dataset llamado 'dataset'
# Realiza una llamada a Optuna para optimizar los hiperparámetros
study = optuna.create_study(directions = ['maximize','maximize','maximize', 'minimize','minimize','minimize'], sampler=None)#None, NSGA por defecto para multiobjetivo, sampler=optuna.samplers.GridSampler(search_space)
print(f"Sampler is {study.sampler.__class__.__name__}")
study.optimize(objective, n_trials=100)#, n_trials=100

# Muestra los mejores hiperparámetros encontrados
print('Best trial:')
trial = study.best_trials
for t in trial:
    print('Value: {}'.format(t.values))
    print('Params: ')
    for key, value in t.params.items():
        print('{}: {}'.format(key, value))

save_study(study, 'prueba')


###hacer tambien lista para las semillas de los train test split en base al numero de folds que haya

c:\Users\da01m\anaconda3\envs\tfg_cnn_lv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2024-04-25 10:05:36,254] A new study created in memory with name: no-name-8bc40b87-20ca-4532-85bb-c77e3ff09131


Sampler is NSGAIISampler
Fold 1
Best epoc: 44. Best val loss: 0.17239366471767426
Accuracy: 0.6666666865348816. Recall: 0.75. Precision: 0.6000000238418579
Fold 2
Best epoc: 24. Best val loss: 0.32758788764476776
Accuracy: 1.0. Recall: 1.0. Precision: 1.0
Fold 3
Best epoc: 23. Best val loss: 0.4073748141527176
Accuracy: 0.6666666865348816. Recall: 0.75. Precision: 0.6000000238418579
Fold 4
Best epoc: 43. Best val loss: 0.1279585249722004
Accuracy: 0.7777777910232544. Recall: 0.5. Precision: 1.0
Fold 5


[I 2024-04-25 10:06:26,601] Trial 0 finished with values: [0.7972222328186035, 0.75, 0.8399999737739563, 0.12777776925460155, 0.15811388194561005, 0.1959591805934906] and parameters: {'batch_size': 5, 'lr': 1.707092743698072e-05, 'data_aug': False, 'ini': 'None', 'e_freeze': 5, 'e_all': 40}. 


Best epoc: 29. Best val loss: 0.3184065595269203
Accuracy: 0.875. Recall: 0.75. Precision: 1.0
Fold 1
Best epoc: 20. Best val loss: 0.5244152098894119
Accuracy: 0.5555555820465088. Recall: 1.0. Precision: 0.5
Fold 2
Best epoc: 39. Best val loss: 0.06850576121360064
Accuracy: 0.7777777910232544. Recall: 0.5. Precision: 1.0
Fold 3
Best epoc: 11. Best val loss: 0.2952309548854828
Accuracy: 1.0. Recall: 1.0. Precision: 1.0
Fold 4
Best epoc: 39. Best val loss: 0.3326272889971733
Accuracy: 0.7777777910232544. Recall: 1.0. Precision: 0.6666666865348816
Fold 5


[I 2024-04-25 10:07:07,376] Trial 1 finished with values: [0.7972222328186035, 0.8999999761581421, 0.7933333516120911, 0.14582670994002656, 0.20000000298023224, 0.1936778575181961] and parameters: {'batch_size': 5, 'lr': 6.619290363939426e-05, 'data_aug': False, 'ini': 'O', 'e_freeze': 3, 'e_all': 37}. 


Best epoc: 13. Best val loss: 0.1307324394583702
Accuracy: 0.875. Recall: 1.0. Precision: 0.800000011920929
Fold 1
Best epoc: 17. Best val loss: 0.313143789768219
Accuracy: 0.6666666865348816. Recall: 0.75. Precision: 0.6000000238418579
Fold 2
Best epoc: 31. Best val loss: 0.3203032910823822
Accuracy: 0.8888888955116272. Recall: 0.75. Precision: 1.0
Fold 3
Best epoc: 47. Best val loss: 0.19466792047023773
Accuracy: 0.7777777910232544. Recall: 0.5. Precision: 1.0
Fold 4
Best epoc: 10. Best val loss: 0.8379242420196533
Accuracy: 0.7777777910232544. Recall: 1.0. Precision: 0.6666666865348816
Fold 5


... (192 outputs left out)

[I 2024-04-25 11:24:09,093] Trial 98 finished with values: [0.8166666746139526, 0.6499999761581421, 0.949999988079071, 0.059835164277754586, 0.12247449159622192, 0.10000000149011612] and parameters: {'batch_size': 5, 'lr': 1.7151690052044462e-05, 'data_aug': True, 'ini': 'O', 'e_freeze': 7, 'e_all': 39}. 


Best epoc: 43. Best val loss: 0.5412985682487488
Accuracy: 0.75. Recall: 0.5. Precision: 1.0
Fold 1
Best epoc: 1. Best val loss: 0.6969661116600037
Accuracy: 0.5555555820465088. Recall: 0.0. Precision: 0.0
Fold 2
Best epoc: 54. Best val loss: 0.13862407207489014
Accuracy: 0.8888888955116272. Recall: 1.0. Precision: 0.800000011920929
Fold 3
Best epoc: 54. Best val loss: 0.012848569080233574
Accuracy: 0.6666666865348816. Recall: 0.75. Precision: 0.6000000238418579
Fold 4
Best epoc: 12. Best val loss: 0.644287109375
Accuracy: 0.8888888955116272. Recall: 0.75. Precision: 1.0
Fold 5


[I 2024-04-25 11:25:02,791] Trial 99 finished with values: [0.725000011920929, 0.550000011920929, 0.6800000071525574, 0.13844372602574564, 0.36742347478866577, 0.37094473956273494] and parameters: {'batch_size': 32, 'lr': 9.660106105900564e-05, 'data_aug': False, 'ini': 'O', 'e_freeze': 6, 'e_all': 49}. 


Best epoc: 54. Best val loss: 0.22718730568885803
Accuracy: 0.625. Recall: 0.25. Precision: 1.0
Best trial:
Value: [0.9111111164093018, 0.8999999761581421, 0.9199999570846558, 0.04444444179534912, 0.12247449159622192, 0.0979795902967453]
Params: 
batch_size: 5
lr: 2.9048016288901607e-05
data_aug: True
ini: H
e_freeze: 4
e_all: 43
Value: [0.8416666746139526, 0.6499999761581421, 1.0, 0.052411002217552065, 0.12247449159622192, 0.0]
Params: 
batch_size: 16
lr: 9.671193332156556e-05
data_aug: False
ini: None
e_freeze: 3
e_all: 36
Value: [0.9305555582046509, 0.8999999761581421, 0.9600000381469727, 0.05692750231639192, 0.12247449159622192, 0.07999999821186066]
Params: 
batch_size: 5
lr: 2.651170581674075e-05
data_aug: True
ini: O
e_freeze: 4
e_all: 45
Value: [0.8861111164093017, 0.75, 1.0, 0.07049209348016061, 0.15811388194561005, 0.0]
Params: 
batch_size: 16
lr: 8.478347526616924e-05
data_aug: False
ini: O
e_freeze: 5
e_all: 46
Value: [0.8861111164093017, 0.800000011920929, 0.960000038146972

In [17]:
seeds = random.sample(range(1,10001), 100)
seeds_print = copy.copy(seeds)

seeds_tr = [random.sample(range(1,10001), 5) for _ in range(100)]
seeds_tr_print = copy.copy(seeds_tr)


##para las semillas de train test split, hago una lista de listas con tantas listas como ejecuciones y cada lista coneteniendo k numeros, según los folds que se hagan
def save_study(study, study_name="study"):
    # Crear una carpeta para guardar el estudio si no existe
    study_dir = os.path.join(os.getcwd(), study_name)
    if not os.path.exists(study_dir):
        os.makedirs(study_dir)
    
    # Guardar los resultados del estudio en un archivo CSV
    study_trials = study.trials_dataframe()
    study_trials.to_csv(os.path.join(study_dir, "study_results.csv"), index=False)

def objective(trial):
    # Define los hiperparámetros para optimizar
    # num_epochs_freeze = trial.suggest_categorical('num_epochs_freeze', [1,3,8,10,15])
    # num_epochs_all = trial.suggest_categorical('num_epochs_all', [25,30,35,40,45,50])
    ##todos estos bs salieron en los mejores, así como ambos extremos del lr
    batch_size = trial.suggest_categorical('batch_size', [16,5])
    lr = trial.suggest_float('lr', 0.001, 0.01)
    data_aug = trial.suggest_categorical('data_aug', [True,False])
    ini = trial.suggest_categorical('ini', ['G', 'H', 'O', 'None'])
    freeze = trial.suggest_int('e_freeze', 3, 7)
    all = trial.suggest_int('e_all', 35, 50)
    
    # optimizer_name = trial.suggest_categorical('optimizer_name', ['RMSprop', 'Adam', 'SGD'])

    # # Convertir el nombre del optimizador en una clase PyTorch
    # if optimizer_name == 'Adam':
    #     optimizer_class = torch.optim.Adam
    # elif optimizer_name == 'SGD':
    #     optimizer_class = torch.optim.SGD
    # elif optimizer_name == 'RMSprop':
    #     optimizer_class = torch.optim.RMSprop

    # Ejecuta la función de entrenamiento con los hiperparámetros seleccionados
    avg_accuracy, avg_recall, avg_precision, acc_std, rec_std, prec_std= prueba_k_fold_resnet_pre(df_mosaic, 5, freeze, all, optim.SGD, batch_size, lr, data_aug, ini, seeds[0], seeds_tr[0])
    seeds.pop(0)
    seeds_tr.pop(0)

    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_pre(df_mean_a_2, 5, 5, 35, optim.Adam, batch_size, lr, data_aug, None)
    # avg_l_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre(df_mean_a_2, 5, 40, optim.Adam, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre_oc(df_mean_a_2, 5, 40, optim.Adam, batch_size, lr, data_aug, None)

    # Devuelve el valor objetivo (en este caso, la pérdida promedio de validación)
    return avg_accuracy, avg_recall, avg_precision, acc_std, rec_std, prec_std

import optuna
# Supongamos que tienes un dataset llamado 'dataset'
# Realiza una llamada a Optuna para optimizar los hiperparámetros
study = optuna.create_study(directions = ['maximize','maximize','maximize', 'minimize','minimize','minimize'], sampler=None)#None, NSGA por defecto para multiobjetivo, sampler=optuna.samplers.GridSampler(search_space)
print(f"Sampler is {study.sampler.__class__.__name__}")
study.optimize(objective, n_trials=100)#, n_trials=100

# Muestra los mejores hiperparámetros encontrados
print('Best trial:')
trial = study.best_trials
for t in trial:
    print('Value: {}'.format(t.values))
    print('Params: ')
    for key, value in t.params.items():
        print('{}: {}'.format(key, value))

save_study(study, 'estduio_tl_sgd')


###hacer tambien lista para las semillas de los train test split en base al numero de folds que haya

[I 2024-04-26 23:14:04,554] A new study created in memory with name: no-name-c8e00266-ea0c-4396-851c-753b9931bcf2


Sampler is NSGAIISampler
Fold 1
Best epoc: 41. Best val loss: 0.46216481924057007
Accuracy: 0.7777777910232544. Recall: 0.75. Precision: 0.75
Fold 2
Best epoc: 43. Best val loss: 0.03348199184983969
Accuracy: 0.8888888955116272. Recall: 0.75. Precision: 1.0
Fold 3
Best epoc: 46. Best val loss: 0.3945990651845932
Accuracy: 0.8888888955116272. Recall: 0.75. Precision: 1.0
Fold 4
Best epoc: 47. Best val loss: 0.2937631793320179
Accuracy: 0.8888888955116272. Recall: 0.75. Precision: 1.0
Fold 5


[I 2024-04-26 23:14:52,244] Trial 0 finished with values: [0.8638888955116272, 0.75, 0.949999988079071, 0.04339027300941767, 0.0, 0.10000000149011612] and parameters: {'batch_size': 5, 'lr': 0.0031666390858920805, 'data_aug': True, 'ini': 'None', 'e_freeze': 7, 'e_all': 42}. 


Best epoc: 18. Best val loss: 0.5062136054039001
Accuracy: 0.875. Recall: 0.75. Precision: 1.0
Fold 1
Best epoc: 37. Best val loss: 0.3305824398994446
Accuracy: 0.7777777910232544. Recall: 0.75. Precision: 0.75
Fold 2
Best epoc: 40. Best val loss: 0.2669143080711365
Accuracy: 0.7777777910232544. Recall: 0.5. Precision: 1.0
Fold 3
Best epoc: 35. Best val loss: 0.5256081223487854
Accuracy: 0.8888888955116272. Recall: 0.75. Precision: 1.0
Fold 4
Best epoc: 39. Best val loss: 0.2780085504055023
Accuracy: 0.8888888955116272. Recall: 1.0. Precision: 0.800000011920929
Fold 5


[I 2024-04-26 23:15:26,528] Trial 1 finished with values: [0.8166666746139526, 0.699999988079071, 0.9100000262260437, 0.059835164277754586, 0.18708287179470062, 0.11135528236627579] and parameters: {'batch_size': 16, 'lr': 0.004353364868276572, 'data_aug': False, 'ini': 'O', 'e_freeze': 7, 'e_all': 35}. 


Best epoc: 24. Best val loss: 0.5018154382705688
Accuracy: 0.75. Recall: 0.5. Precision: 1.0
Fold 1
Best epoc: 48. Best val loss: 0.22767139971256256
Accuracy: 0.7777777910232544. Recall: 0.75. Precision: 0.75
Fold 2
Best epoc: 44. Best val loss: 0.4195539653301239
Accuracy: 0.7777777910232544. Recall: 0.75. Precision: 0.75
Fold 3
Best epoc: 34. Best val loss: 0.5509799122810364
Accuracy: 1.0. Recall: 1.0. Precision: 1.0
Fold 4
Best epoc: 48. Best val loss: 0.34147754311561584
Accuracy: 0.5555555820465088. Recall: 0.5. Precision: 0.5
Fold 5


... (192 outputs left out)

[I 2024-04-27 00:23:43,663] Trial 98 finished with values: [0.9083333373069763, 0.800000011920929, 1.0, 0.08407080641548241, 0.18708285689353943, 0.0] and parameters: {'batch_size': 5, 'lr': 0.005193424795577692, 'data_aug': False, 'ini': 'None', 'e_freeze': 5, 'e_all': 38}. 


Best epoc: 35. Best val loss: 0.11309177055954933
Accuracy: 0.875. Recall: 0.75. Precision: 1.0
Fold 1
Best epoc: 34. Best val loss: 0.6316229104995728
Accuracy: 0.6666666865348816. Recall: 0.25. Precision: 1.0
Fold 2
Best epoc: 47. Best val loss: 0.41225606203079224
Accuracy: 0.6666666865348816. Recall: 0.5. Precision: 0.6666666865348816
Fold 3
Best epoc: 49. Best val loss: 0.386691153049469
Accuracy: 1.0. Recall: 1.0. Precision: 1.0
Fold 4
Best epoc: 51. Best val loss: 0.27036434412002563
Accuracy: 0.8888888955116272. Recall: 1.0. Precision: 0.800000011920929
Fold 5


[I 2024-04-27 00:24:28,391] Trial 99 finished with values: [0.819444453716278, 0.699999988079071, 0.8933333158493042, 0.13205403954656564, 0.29154759645462036, 0.13727505505084991] and parameters: {'batch_size': 16, 'lr': 0.0015418821297510494, 'data_aug': True, 'ini': 'O', 'e_freeze': 5, 'e_all': 49}. 


Best epoc: 53. Best val loss: 0.4356067180633545
Accuracy: 0.875. Recall: 0.75. Precision: 1.0
Best trial:
Value: [0.8638888955116272, 0.75, 0.949999988079071, 0.04339027300941767, 0.0, 0.10000000149011612]
Params: 
batch_size: 5
lr: 0.0031666390858920805
data_aug: True
ini: None
e_freeze: 7
e_all: 42
Value: [0.9305555582046509, 0.8500000238418579, 1.0, 0.05692750231639192, 0.12247449159622192, 0.0]
Params: 
batch_size: 5
lr: 0.0092563178193737
data_aug: True
ini: H
e_freeze: 7
e_all: 47
Value: [0.8666666746139526, 0.800000011920929, 0.8999999761581421, 0.08314793697230627, 0.10000000149011612, 0.12247449159622192]
Params: 
batch_size: 5
lr: 0.007968355177105979
data_aug: False
ini: H
e_freeze: 4
e_all: 36
Value: [0.8638888955116272, 0.800000011920929, 0.9100000262260437, 0.04339027300941767, 0.10000000149011612, 0.11135528236627579]
Params: 
batch_size: 5
lr: 0.004746181706783053
data_aug: True
ini: H
e_freeze: 5
e_all: 47
Value: [0.8638888955116272, 0.8999999761581421, 0.874285697937

In [6]:
seeds = random.sample(range(1,10001), 100)
seeds_print = copy.copy(seeds)

seeds_tr = [random.sample(range(1,10001), 5) for _ in range(100)]
seeds_tr_print = copy.copy(seeds_tr)
def save_study(study, study_name="study"):
    # Crear una carpeta para guardar el estudio si no existe
    study_dir = os.path.join(os.getcwd(), study_name)
    if not os.path.exists(study_dir):
        os.makedirs(study_dir)
    
    # Guardar los resultados del estudio en un archivo CSV
    study_trials = study.trials_dataframe()
    study_trials.to_csv(os.path.join(study_dir, "study_results.csv"), index=False)

def objective(trial):
    # Define los hiperparámetros para optimizar
    # num_epochs_freeze = trial.suggest_categorical('num_epochs_freeze', [1,3,8,10,15])
    # num_epochs_all = trial.suggest_categorical('num_epochs_all', [25,30,35,40,45,50])
    ### SOLO DIO UNA CON 5, SE METE 16 POR PROBAR, AL IGULA QUE EL LR
    batch_size = trial.suggest_categorical('batch_size', [16,5])
    lr = trial.suggest_float('lr', 1e-5/2, 1e-5*2)
    data_aug = trial.suggest_categorical('data_aug', [True,False])
    ini = trial.suggest_categorical('ini', ['G', 'H', 'O', 'None'])
    freeze = trial.suggest_int('e_freeze', 3, 7)
    all = trial.suggest_int('e_all', 35, 50)
    
    # optimizer_name = trial.suggest_categorical('optimizer_name', ['RMSprop', 'Adam', 'SGD'])

    # # Convertir el nombre del optimizador en una clase PyTorch
    # if optimizer_name == 'Adam':
    #     optimizer_class = torch.optim.Adam
    # elif optimizer_name == 'SGD':
    #     optimizer_class = torch.optim.SGD
    # elif optimizer_name == 'RMSprop':
    #     optimizer_class = torch.optim.RMSprop

    # Ejecuta la función de entrenamiento con los hiperparámetros seleccionados
    avg_accuracy, avg_recall, avg_precision, acc_std, rec_std, prec_std= prueba_k_fold_resnet_pre(df_mosaic, 5, freeze, all, optim.RMSprop, batch_size, lr, data_aug, ini, seeds[0], seeds_tr[0])
    seeds.pop(0)
    seeds_tr.pop(0)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_pre(df_mean_a_2, 5, 5, 35, optim.Adam, batch_size, lr, data_aug, None)
    # avg_l_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre(df_mean_a_2, 5, 40, optim.Adam, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre_oc(df_mean_a_2, 5, 40, optim.Adam, batch_size, lr, data_aug, None)

    # Devuelve el valor objetivo (en este caso, la pérdida promedio de validación)
    return avg_accuracy, avg_recall, avg_precision, acc_std, rec_std, prec_std

import optuna
# Supongamos que tienes un dataset llamado 'dataset'
# Realiza una llamada a Optuna para optimizar los hiperparámetros
study = optuna.create_study(directions = ['maximize','maximize','maximize', 'minimize','minimize','minimize'], sampler=None)#None, NSGA por defecto para multiobjetivo, sampler=optuna.samplers.GridSampler(search_space)
print(f"Sampler is {study.sampler.__class__.__name__}")
study.optimize(objective, n_trials=100)#, n_trials=100

# Muestra los mejores hiperparámetros encontrados
print('Best trial:')
trial = study.best_trials
for t in trial:
    print('Value: {}'.format(t.values))
    print('Params: ')
    for key, value in t.params.items():
        print('{}: {}'.format(key, value))

save_study(study, 'prueba_2')

[I 2024-04-25 11:25:02,888] A new study created in memory with name: no-name-5faa4581-d3d1-4f54-a2f0-841c9286afa1


Sampler is NSGAIISampler
Fold 1
Best epoc: 9. Best val loss: 0.6202006340026855
Accuracy: 1.0. Recall: 1.0. Precision: 1.0
Fold 2
Best epoc: 52. Best val loss: 0.2157844752073288
Accuracy: 0.8888888955116272. Recall: 1.0. Precision: 0.800000011920929
Fold 3
Best epoc: 50. Best val loss: 0.20505717396736145
Accuracy: 0.7777777910232544. Recall: 0.5. Precision: 1.0
Fold 4
Best epoc: 46. Best val loss: 0.25230711698532104
Accuracy: 0.7777777910232544. Recall: 1.0. Precision: 0.6666666865348816
Fold 5


[I 2024-04-25 11:25:48,985] Trial 0 finished with values: [0.8638888955116272, 0.8500000238418579, 0.8933333158493042, 0.08258926569525993, 0.20000000298023224, 0.1372750699520111] and parameters: {'batch_size': 16, 'lr': 9.012284076989228e-06, 'data_aug': True, 'ini': 'None', 'e_freeze': 3, 'e_all': 50}. 


Best epoc: 31. Best val loss: 0.40768519043922424
Accuracy: 0.875. Recall: 0.75. Precision: 1.0
Fold 1
Best epoc: 32. Best val loss: 0.4103318154811859
Accuracy: 1.0. Recall: 1.0. Precision: 1.0
Fold 2
Best epoc: 8. Best val loss: 0.5000293254852295
Accuracy: 1.0. Recall: 1.0. Precision: 1.0
Fold 3
Best epoc: 6. Best val loss: 0.6242733001708984
Accuracy: 0.4444444477558136. Recall: 0.75. Precision: 0.4285714328289032
Fold 4
Best epoc: 31. Best val loss: 0.5264144539833069
Accuracy: 0.7777777910232544. Recall: 0.5. Precision: 1.0
Fold 5


[I 2024-04-25 11:26:27,543] Trial 1 finished with values: [0.8194444477558136, 0.800000011920929, 0.8857143521308899, 0.20525503382905458, 0.18708285689353943, 0.22857141494750977] and parameters: {'batch_size': 16, 'lr': 8.995876260315175e-06, 'data_aug': True, 'ini': 'H', 'e_freeze': 5, 'e_all': 38}. 


Best epoc: 42. Best val loss: 0.07714637368917465
Accuracy: 0.875. Recall: 0.75. Precision: 1.0
Fold 1
Best epoc: 46. Best val loss: 0.24154414236545563
Accuracy: 0.8888888955116272. Recall: 0.75. Precision: 1.0
Fold 2
Best epoc: 28. Best val loss: 0.22999346256256104
Accuracy: 0.7777777910232544. Recall: 0.5. Precision: 1.0
Fold 3
Best epoc: 10. Best val loss: 0.5260657072067261
Accuracy: 0.7777777910232544. Recall: 0.75. Precision: 0.75
Fold 4
Best epoc: 40. Best val loss: 0.3342653512954712
Accuracy: 0.7777777910232544. Recall: 0.75. Precision: 0.75
Fold 5


... (192 outputs left out)

[I 2024-04-25 12:40:15,588] Trial 98 finished with values: [0.8388888955116272, 0.8500000238418579, 0.8300000429153442, 0.1170628147616229, 0.12247449159622192, 0.15362291038036346] and parameters: {'batch_size': 5, 'lr': 6.78434799372939e-06, 'data_aug': False, 'ini': 'None', 'e_freeze': 3, 'e_all': 39}. 


Best epoc: 36. Best val loss: 0.21379265189170837
Accuracy: 0.75. Recall: 0.75. Precision: 0.75
Fold 1
Best epoc: 37. Best val loss: 0.10542803769931197
Accuracy: 0.7777777910232544. Recall: 0.75. Precision: 0.75
Fold 2
Best epoc: 18. Best val loss: 0.3810741901397705
Accuracy: 0.6666666865348816. Recall: 0.75. Precision: 0.6000000238418579
Fold 3
Best epoc: 46. Best val loss: 0.05093883536756039
Accuracy: 0.8888888955116272. Recall: 0.75. Precision: 1.0
Fold 4
Best epoc: 46. Best val loss: 0.007980131777003407
Accuracy: 0.8888888955116272. Recall: 0.75. Precision: 1.0
Fold 5


[I 2024-04-25 12:41:01,936] Trial 99 finished with values: [0.819444453716278, 0.75, 0.8700000047683716, 0.08695819298987903, 0.0, 0.16613247990608215] and parameters: {'batch_size': 5, 'lr': 1.7352557638565057e-05, 'data_aug': False, 'ini': 'None', 'e_freeze': 7, 'e_all': 41}. 


Best epoc: 45. Best val loss: 0.17986436188220978
Accuracy: 0.875. Recall: 0.75. Precision: 1.0
Best trial:
Value: [0.8638888955116272, 0.8500000238418579, 0.8933333158493042, 0.04339027300941767, 0.12247449159622192, 0.13727505505084991]
Params: 
batch_size: 16
lr: 1.822072180539651e-05
data_aug: True
ini: G
e_freeze: 4
e_all: 41
Value: [0.9083333373069763, 0.8500000238418579, 0.9600000381469727, 0.04614790867524854, 0.12247449159622192, 0.07999999821186066]
Params: 
batch_size: 16
lr: 1.9536705471586094e-05
data_aug: True
ini: O
e_freeze: 5
e_all: 46
Value: [0.8638888955116272, 0.699999988079071, 1.0, 0.04339027300941767, 0.10000000149011612, 0.0]
Params: 
batch_size: 16
lr: 9.134151384783784e-06
data_aug: True
ini: None
e_freeze: 6
e_all: 43
Value: [0.8861111164093017, 0.800000011920929, 0.9600000381469727, 0.005555558204650879, 0.10000000149011612, 0.07999999821186066]
Params: 
batch_size: 16
lr: 9.6682518967905e-06
data_aug: True
ini: None
e_freeze: 4
e_all: 48
Value: [0.933333337

## PRUEBA CON TL Y OC

In [6]:
seeds = random.sample(range(1,10001), 100)
seeds_print = copy.copy(seeds)

seeds_tr = [random.sample(range(1,10001), 5) for _ in range(100)]
seeds_tr_print = copy.copy(seeds_tr)


##para las semillas de train test split, hago una lista de listas con tantas listas como ejecuciones y cada lista coneteniendo k numeros, según los folds que se hagan
def save_study(study, study_name="study"):
    # Crear una carpeta para guardar el estudio si no existe
    study_dir = os.path.join(os.getcwd(), study_name)
    if not os.path.exists(study_dir):
        os.makedirs(study_dir)
    
    # Guardar los resultados del estudio en un archivo CSV
    study_trials = study.trials_dataframe()
    study_trials.to_csv(os.path.join(study_dir, "study_results.csv"), index=False)

def objective(trial):
    # Define los hiperparámetros para optimizar
    # num_epochs_freeze = trial.suggest_categorical('num_epochs_freeze', [1,3,8,10,15])
    # num_epochs_all = trial.suggest_categorical('num_epochs_all', [25,30,35,40,45,50])
    ##todos estos bs salieron en los mejores, así como ambos extremos del lr
    batch_size = trial.suggest_categorical('batch_size', [16,5,32])
    lr = trial.suggest_float('lr', 0.001/2, 0.001*2)
    data_aug = trial.suggest_categorical('data_aug', [True,False])
    ini = trial.suggest_categorical('ini', ['G', 'H', 'O', 'None'])
    freeze = trial.suggest_int('e_freeze', 3, 7)
    all = trial.suggest_int('e_all', 35, 50)
    
    # optimizer_name = trial.suggest_categorical('optimizer_name', ['RMSprop', 'Adam', 'SGD'])

    # # Convertir el nombre del optimizador en una clase PyTorch
    # if optimizer_name == 'Adam':
    #     optimizer_class = torch.optim.Adam
    # elif optimizer_name == 'SGD':
    #     optimizer_class = torch.optim.SGD
    # elif optimizer_name == 'RMSprop':
    #     optimizer_class = torch.optim.RMSprop

    # Ejecuta la función de entrenamiento con los hiperparámetros seleccionados
    avg_accuracy, avg_recall, avg_precision, acc_std, rec_std, prec_std= prueba_k_fold_resnet_pre_oc(df_mosaic, 5, freeze, all, optim.Adam, batch_size, lr, data_aug, ini, seeds[0], seeds_tr[0])
    seeds.pop(0)
    seeds_tr.pop(0)

    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_pre(df_mean_a_2, 5, 5, 35, optim.Adam, batch_size, lr, data_aug, None)
    # avg_l_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre(df_mean_a_2, 5, 40, optim.Adam, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre_oc(df_mean_a_2, 5, 40, optim.Adam, batch_size, lr, data_aug, None)

    # Devuelve el valor objetivo (en este caso, la pérdida promedio de validación)
    return avg_accuracy, avg_recall, avg_precision, acc_std, rec_std, prec_std

import optuna
# Supongamos que tienes un dataset llamado 'dataset'
# Realiza una llamada a Optuna para optimizar los hiperparámetros
study = optuna.create_study(directions = ['maximize','maximize','maximize', 'minimize','minimize','minimize'], sampler=None)#None, NSGA por defecto para multiobjetivo, sampler=optuna.samplers.GridSampler(search_space)
print(f"Sampler is {study.sampler.__class__.__name__}")
study.optimize(objective, n_trials=100)#, n_trials=100

# Muestra los mejores hiperparámetros encontrados
print('Best trial:')
trial = study.best_trials
for t in trial:
    print('Value: {}'.format(t.values))
    print('Params: ')
    for key, value in t.params.items():
        print('{}: {}'.format(key, value))

save_study(study, 'estudio_tl_oc_adam')


###hacer tambien lista para las semillas de los train test split en base al numero de folds que haya

c:\Users\da01m\anaconda3\envs\tfg_cnn_lv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2024-04-27 10:46:46,956] A new study created in memory with name: no-name-09ef257a-bd97-4ac6-9abb-3f1695c0a496


Sampler is NSGAIISampler
Fold 1
Best epoc: 5. Best val loss: 0.6401274800300598
Accuracy: 0.4444444477558136. Recall: 0.25. Precision: 0.3333333432674408
Fold 2
Best epoc: 51. Best val loss: 0.4270095229148865
Accuracy: 0.7777777910232544. Recall: 0.5. Precision: 1.0
Fold 3
Best epoc: 43. Best val loss: 0.5042825937271118
Accuracy: 0.5555555820465088. Recall: 1.0. Precision: 0.5
Fold 4
Best epoc: 3. Best val loss: 0.648034930229187
Accuracy: 0.5555555820465088. Recall: 1.0. Precision: 0.5
Fold 5


[I 2024-04-27 10:47:40,599] Trial 0 finished with values: [0.5666666805744172, 0.75, 0.5666667222976685, 0.11331154792863014, 0.3162277638912201, 0.22607766091823578] and parameters: {'batch_size': 16, 'lr': 0.0016270841556829, 'data_aug': True, 'ini': 'H', 'e_freeze': 6, 'e_all': 50}. 


Best epoc: 48. Best val loss: 0.8501607179641724
Accuracy: 0.5. Recall: 1.0. Precision: 0.5
Fold 1
Best epoc: 39. Best val loss: 0.14690001308918
Accuracy: 0.7777777910232544. Recall: 0.75. Precision: 0.75
Fold 2
Best epoc: 38. Best val loss: 0.4067441523075104
Accuracy: 0.7777777910232544. Recall: 0.5. Precision: 1.0
Fold 3
Best epoc: 35. Best val loss: 0.7861615121364594
Accuracy: 0.8888888955116272. Recall: 0.75. Precision: 1.0
Fold 4
Best epoc: 43. Best val loss: 0.7535329461097717
Accuracy: 0.5555555820465088. Recall: 0.75. Precision: 0.5
Fold 5


[I 2024-04-27 10:48:23,903] Trial 1 finished with values: [0.6500000119209289, 0.6000000238418579, 0.699999988079071, 0.22743877428393916, 0.20000000298023224, 0.29154759645462036] and parameters: {'batch_size': 5, 'lr': 0.0010773529701312155, 'data_aug': False, 'ini': 'H', 'e_freeze': 3, 'e_all': 41}. 


Best epoc: 0. Best val loss: 0.8190735578536987
Accuracy: 0.25. Recall: 0.25. Precision: 0.25
Fold 1
Best epoc: 44. Best val loss: 0.4703064262866974
Accuracy: 0.7777777910232544. Recall: 0.75. Precision: 0.75
Fold 2
Best epoc: 48. Best val loss: 0.4924132823944092
Accuracy: 0.8888888955116272. Recall: 0.75. Precision: 1.0
Fold 3
Best epoc: 0. Best val loss: 0.6383742094039917
Accuracy: 0.6666666865348816. Recall: 0.25. Precision: 1.0
Fold 4
Best epoc: 47. Best val loss: 0.6417635679244995
Accuracy: 0.5555555820465088. Recall: 0.25. Precision: 0.5
Fold 5


... (192 outputs left out)

[I 2024-04-27 12:10:20,303] Trial 98 finished with values: [0.6166666805744171, 0.30000001192092896, 0.45, 0.12717247672566787, 0.29154759645462036, 0.4] and parameters: {'batch_size': 16, 'lr': 0.0010773529701312155, 'data_aug': True, 'ini': 'None', 'e_freeze': 7, 'e_all': 49}. 


Best epoc: 0. Best val loss: 0.7486832141876221
Accuracy: 0.75. Recall: 0.75. Precision: 0.75
Fold 1
Best epoc: 49. Best val loss: 0.44976896047592163
Accuracy: 0.6666666865348816. Recall: 0.25. Precision: 1.0
Fold 2
Best epoc: 39. Best val loss: 0.4341052770614624
Accuracy: 0.8888888955116272. Recall: 1.0. Precision: 0.800000011920929
Fold 3
Best epoc: 48. Best val loss: 0.6016616821289062
Accuracy: 0.6666666865348816. Recall: 0.25. Precision: 1.0
Fold 4
Best epoc: 37. Best val loss: 0.792667031288147
Accuracy: 0.5555555820465088. Recall: 0.75. Precision: 0.5
Fold 5


[I 2024-04-27 12:11:07,613] Trial 99 finished with values: [0.6555555701255799, 0.6499999761581421, 0.7599999904632568, 0.13333333233992295, 0.33911648392677307, 0.22449944913387299] and parameters: {'batch_size': 32, 'lr': 0.0019314174569345262, 'data_aug': True, 'ini': 'H', 'e_freeze': 5, 'e_all': 45}. 


Best epoc: 12. Best val loss: 0.6686893701553345
Accuracy: 0.5. Recall: 1.0. Precision: 0.5
Best trial:
Value: [0.5666666805744172, 0.75, 0.5666667222976685, 0.11331154792863014, 0.3162277638912201, 0.22607766091823578]
Params: 
batch_size: 16
lr: 0.0016270841556829
data_aug: True
ini: H
e_freeze: 6
e_all: 50
Value: [0.6333333492279053, 0.6000000238418579, 0.6000000238418579, 0.11967032855550919, 0.29999998211860657, 0.12247449159622192]
Params: 
batch_size: 5
lr: 0.0011741213647364248
data_aug: True
ini: H
e_freeze: 3
e_all: 49
Value: [0.6805555701255799, 0.6000000238418579, 0.7833333611488342, 0.08695819679788047, 0.25495097041130066, 0.19436505436897278]
Params: 
batch_size: 16
lr: 0.0019207960550477814
data_aug: False
ini: H
e_freeze: 7
e_all: 40
Value: [0.5472222328186035, 0.8999999761581421, 0.49841269850730896, 0.13482957115612748, 0.20000000298023224, 0.09608093649148941]
Params: 
batch_size: 32
lr: 0.0005868485938333585
data_aug: False
ini: G
e_freeze: 5
e_all: 36
Value: [0.70

In [7]:
seeds = random.sample(range(1,10001), 100)
seeds_print = copy.copy(seeds)

seeds_tr = [random.sample(range(1,10001), 5) for _ in range(100)]
seeds_tr_print = copy.copy(seeds_tr)


##para las semillas de train test split, hago una lista de listas con tantas listas como ejecuciones y cada lista coneteniendo k numeros, según los folds que se hagan
def save_study(study, study_name="study"):
    # Crear una carpeta para guardar el estudio si no existe
    study_dir = os.path.join(os.getcwd(), study_name)
    if not os.path.exists(study_dir):
        os.makedirs(study_dir)
    
    # Guardar los resultados del estudio en un archivo CSV
    study_trials = study.trials_dataframe()
    study_trials.to_csv(os.path.join(study_dir, "study_results.csv"), index=False)

def objective(trial):
    # Define los hiperparámetros para optimizar
    # num_epochs_freeze = trial.suggest_categorical('num_epochs_freeze', [1,3,8,10,15])
    # num_epochs_all = trial.suggest_categorical('num_epochs_all', [25,30,35,40,45,50])
    ##todos estos bs salieron en los mejores, así como ambos extremos del lr
    batch_size = trial.suggest_categorical('batch_size', [16,5])
    lr = trial.suggest_float('lr', 0.0001*5, 0.01)
    data_aug = trial.suggest_categorical('data_aug', [True,False])
    ini = trial.suggest_categorical('ini', ['G', 'H', 'O', 'None'])
    freeze = trial.suggest_int('e_freeze', 3, 7)
    all = trial.suggest_int('e_all', 35, 50)
    
    # optimizer_name = trial.suggest_categorical('optimizer_name', ['RMSprop', 'Adam', 'SGD'])

    # # Convertir el nombre del optimizador en una clase PyTorch
    # if optimizer_name == 'Adam':
    #     optimizer_class = torch.optim.Adam
    # elif optimizer_name == 'SGD':
    #     optimizer_class = torch.optim.SGD
    # elif optimizer_name == 'RMSprop':
    #     optimizer_class = torch.optim.RMSprop

    # Ejecuta la función de entrenamiento con los hiperparámetros seleccionados
    avg_accuracy, avg_recall, avg_precision, acc_std, rec_std, prec_std= prueba_k_fold_resnet_pre_oc(df_mosaic, 5, freeze, all, optim.SGD, batch_size, lr, data_aug, ini, seeds[0], seeds_tr[0])
    seeds.pop(0)
    seeds_tr.pop(0)

    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_pre(df_mean_a_2, 5, 5, 35, optim.Adam, batch_size, lr, data_aug, None)
    # avg_l_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre(df_mean_a_2, 5, 40, optim.Adam, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre_oc(df_mean_a_2, 5, 40, optim.Adam, batch_size, lr, data_aug, None)

    # Devuelve el valor objetivo (en este caso, la pérdida promedio de validación)
    return avg_accuracy, avg_recall, avg_precision, acc_std, rec_std, prec_std

import optuna
# Supongamos que tienes un dataset llamado 'dataset'
# Realiza una llamada a Optuna para optimizar los hiperparámetros
study = optuna.create_study(directions = ['maximize','maximize','maximize', 'minimize','minimize','minimize'], sampler=None)#None, NSGA por defecto para multiobjetivo, sampler=optuna.samplers.GridSampler(search_space)
print(f"Sampler is {study.sampler.__class__.__name__}")
study.optimize(objective, n_trials=100)#, n_trials=100

# Muestra los mejores hiperparámetros encontrados
print('Best trial:')
trial = study.best_trials
for t in trial:
    print('Value: {}'.format(t.values))
    print('Params: ')
    for key, value in t.params.items():
        print('{}: {}'.format(key, value))

save_study(study, 'estudio_tl_oc_sgd')


###hacer tambien lista para las semillas de los train test split en base al numero de folds que haya

[I 2024-04-27 12:11:07,782] A new study created in memory with name: no-name-74bb6c09-ae2c-4f71-8e83-507c9a3519b9


Sampler is NSGAIISampler
Fold 1
Best epoc: 0. Best val loss: 0.6080224514007568
Accuracy: 0.7777777910232544. Recall: 0.75. Precision: 0.75
Fold 2
Best epoc: 25. Best val loss: 0.48010456562042236
Accuracy: 0.5555555820465088. Recall: 0.25. Precision: 0.5
Fold 3
Best epoc: 4. Best val loss: 0.7239832282066345
Accuracy: 0.6666666865348816. Recall: 0.5. Precision: 0.6666666865348816
Fold 4
Best epoc: 29. Best val loss: 0.5775939226150513
Accuracy: 0.6666666865348816. Recall: 0.25. Precision: 1.0
Fold 5


[I 2024-04-27 12:11:52,665] Trial 0 finished with values: [0.6583333492279053, 0.4000000059604645, 0.7833333611488342, 0.07222221998068055, 0.20000000298023224, 0.19436505436897278] and parameters: {'batch_size': 16, 'lr': 0.0037500966976866537, 'data_aug': True, 'ini': 'O', 'e_freeze': 6, 'e_all': 49}. 


Best epoc: 49. Best val loss: 0.7269603610038757
Accuracy: 0.625. Recall: 0.25. Precision: 1.0
Fold 1
Best epoc: 44. Best val loss: 0.6376298666000366
Accuracy: 0.5555555820465088. Recall: 0.5. Precision: 0.5
Fold 2
Best epoc: 6. Best val loss: 0.7548201680183411
Accuracy: 0.7777777910232544. Recall: 1.0. Precision: 0.6666666865348816
Fold 3
Best epoc: 12. Best val loss: 0.48180627822875977
Accuracy: 0.4444444477558136. Recall: 0.25. Precision: 0.3333333432674408
Fold 4
Best epoc: 3. Best val loss: 0.7053049206733704
Accuracy: 0.5555555820465088. Recall: 0.25. Precision: 0.5
Fold 5


[I 2024-04-27 12:12:30,609] Trial 1 finished with values: [0.5666666805744172, 0.4000000059604645, 0.4000000059604645, 0.11331154792863014, 0.33911651372909546, 0.22607767020534758] and parameters: {'batch_size': 16, 'lr': 0.004482472596540775, 'data_aug': True, 'ini': 'G', 'e_freeze': 4, 'e_all': 42}. 


Best epoc: 3. Best val loss: 0.7365444898605347
Accuracy: 0.5. Recall: 0.0. Precision: 0.0
Fold 1
Best epoc: 11. Best val loss: 0.6419507265090942
Accuracy: 0.7777777910232544. Recall: 0.5. Precision: 1.0
Fold 2
Best epoc: 42. Best val loss: 0.5807653665542603
Accuracy: 0.8888888955116272. Recall: 0.75. Precision: 1.0
Fold 3
Best epoc: 43. Best val loss: 0.614842414855957
Accuracy: 0.5555555820465088. Recall: 0.25. Precision: 0.5
Fold 4
Best epoc: 0. Best val loss: 0.7405111789703369
Accuracy: 0.5555555820465088. Recall: 0.0. Precision: 0.0
Fold 5


... (192 outputs left out)

[I 2024-04-27 13:23:24,453] Trial 98 finished with values: [0.7166666746139526, 0.6000000238418579, 0.7833333611488342, 0.2584378529539303, 0.25495097041130066, 0.2962731420993805] and parameters: {'batch_size': 5, 'lr': 0.009343466668689933, 'data_aug': False, 'ini': 'G', 'e_freeze': 5, 'e_all': 44}. 


Best epoc: 0. Best val loss: 0.7290426194667816
Accuracy: 0.25. Recall: 0.25. Precision: 0.25
Fold 1
Best epoc: 49. Best val loss: 0.3651111274957657
Accuracy: 0.8888888955116272. Recall: 1.0. Precision: 0.800000011920929
Fold 2
Best epoc: 50. Best val loss: 0.32242903113365173
Accuracy: 0.7777777910232544. Recall: 0.5. Precision: 1.0
Fold 3
Best epoc: 45. Best val loss: 0.4383452981710434
Accuracy: 0.5555555820465088. Recall: 0.5. Precision: 0.5
Fold 4
Best epoc: 51. Best val loss: 0.4051854908466339
Accuracy: 0.7777777910232544. Recall: 0.5. Precision: 1.0
Fold 5


[I 2024-04-27 13:24:17,690] Trial 99 finished with values: [0.725000011920929, 0.550000011920929, 0.8600000143051147, 0.1192828330466764, 0.24494896829128265, 0.1959591805934906] and parameters: {'batch_size': 5, 'lr': 0.00862138938560251, 'data_aug': True, 'ini': 'H', 'e_freeze': 4, 'e_all': 49}. 


Best epoc: 1. Best val loss: 0.5014625787734985
Accuracy: 0.625. Recall: 0.25. Precision: 1.0
Best trial:
Value: [0.7027777910232544, 0.6499999761581421, 0.7076190710067749, 0.11772010760588167, 0.29999998211860657, 0.177398681640625]
Params: 
batch_size: 5
lr: 0.007843306343585365
data_aug: True
ini: H
e_freeze: 6
e_all: 46
Value: [0.6833333432674408, 0.6000000238418579, 0.8488888740539551, 0.14444444431708434, 0.33911648392677307, 0.2165498286485672]
Params: 
batch_size: 16
lr: 0.00850864554546372
data_aug: True
ini: O
e_freeze: 6
e_all: 41
Value: [0.5888889014720917, 0.550000011920929, 0.5866667032241821, 0.1196703335362462, 0.18708285689353943, 0.22370615601539612]
Params: 
batch_size: 16
lr: 0.0023657048037676936
data_aug: True
ini: G
e_freeze: 7
e_all: 42
Value: [0.6361111223697662, 0.6499999761581421, 0.6922222375869751, 0.10844012284035913, 0.25495097041130066, 0.18363948166370392]
Params: 
batch_size: 5
lr: 0.0014933004980756445
data_aug: True
ini: H
e_freeze: 4
e_all: 45
Valu

In [8]:
seeds = random.sample(range(1,10001), 100)
seeds_print = copy.copy(seeds)

seeds_tr = [random.sample(range(1,10001), 5) for _ in range(100)]
seeds_tr_print = copy.copy(seeds_tr)


##para las semillas de train test split, hago una lista de listas con tantas listas como ejecuciones y cada lista coneteniendo k numeros, según los folds que se hagan
def save_study(study, study_name="study"):
    # Crear una carpeta para guardar el estudio si no existe
    study_dir = os.path.join(os.getcwd(), study_name)
    if not os.path.exists(study_dir):
        os.makedirs(study_dir)
    
    # Guardar los resultados del estudio en un archivo CSV
    study_trials = study.trials_dataframe()
    study_trials.to_csv(os.path.join(study_dir, "study_results.csv"), index=False)

def objective(trial):
    # Define los hiperparámetros para optimizar
    # num_epochs_freeze = trial.suggest_categorical('num_epochs_freeze', [1,3,8,10,15])
    # num_epochs_all = trial.suggest_categorical('num_epochs_all', [25,30,35,40,45,50])
    ##todos estos bs salieron en los mejores, así como ambos extremos del lr
    batch_size = trial.suggest_categorical('batch_size', [16,5,32])
    lr = trial.suggest_float('lr', 1e-5, 0.0001)
    data_aug = trial.suggest_categorical('data_aug', [True,False])
    ini = trial.suggest_categorical('ini', ['G', 'H', 'O', 'None'])
    freeze = trial.suggest_int('e_freeze', 3, 7)
    all = trial.suggest_int('e_all', 35, 50)
    
    # optimizer_name = trial.suggest_categorical('optimizer_name', ['RMSprop', 'Adam', 'SGD'])

    # # Convertir el nombre del optimizador en una clase PyTorch
    # if optimizer_name == 'Adam':
    #     optimizer_class = torch.optim.Adam
    # elif optimizer_name == 'SGD':
    #     optimizer_class = torch.optim.SGD
    # elif optimizer_name == 'RMSprop':
    #     optimizer_class = torch.optim.RMSprop

    # Ejecuta la función de entrenamiento con los hiperparámetros seleccionados
    avg_accuracy, avg_recall, avg_precision, acc_std, rec_std, prec_std= prueba_k_fold_resnet_pre_oc(df_mosaic, 5, freeze, all, optim.RMSprop, batch_size, lr, data_aug, ini, seeds[0], seeds_tr[0])
    seeds.pop(0)
    seeds_tr.pop(0)

    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_pre(df_mean_a_2, 5, 5, 35, optim.Adam, batch_size, lr, data_aug, None)
    # avg_l_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre(df_mean_a_2, 5, 40, optim.Adam, batch_size, lr, data_aug, None)
    # avg_val_loss,avg_val_accuracy, avg_val_recall, avg_val_precision = prueba_k_fold_resnet_no_pre_oc(df_mean_a_2, 5, 40, optim.Adam, batch_size, lr, data_aug, None)

    # Devuelve el valor objetivo (en este caso, la pérdida promedio de validación)
    return avg_accuracy, avg_recall, avg_precision, acc_std, rec_std, prec_std

import optuna
# Supongamos que tienes un dataset llamado 'dataset'
# Realiza una llamada a Optuna para optimizar los hiperparámetros
study = optuna.create_study(directions = ['maximize','maximize','maximize', 'minimize','minimize','minimize'], sampler=None)#None, NSGA por defecto para multiobjetivo, sampler=optuna.samplers.GridSampler(search_space)
print(f"Sampler is {study.sampler.__class__.__name__}")
study.optimize(objective, n_trials=100)#, n_trials=100

# Muestra los mejores hiperparámetros encontrados
print('Best trial:')
trial = study.best_trials
for t in trial:
    print('Value: {}'.format(t.values))
    print('Params: ')
    for key, value in t.params.items():
        print('{}: {}'.format(key, value))

save_study(study, 'estudio_tl_oc_rms')


###hacer tambien lista para las semillas de los train test split en base al numero de folds que haya

[I 2024-04-27 13:24:17,761] A new study created in memory with name: no-name-02bd926d-1005-4bbb-8073-ef3802bfd625


Sampler is NSGAIISampler
Fold 1
Best epoc: 41. Best val loss: 0.555471658706665
Accuracy: 0.5555555820465088. Recall: 0.25. Precision: 0.5
Fold 2
Best epoc: 23. Best val loss: 0.6794293522834778
Accuracy: 0.6666666865348816. Recall: 0.5. Precision: 0.6666666865348816
Fold 3
Best epoc: 35. Best val loss: 0.48333075642585754
Accuracy: 0.7777777910232544. Recall: 1.0. Precision: 0.6666666865348816
Fold 4
Best epoc: 6. Best val loss: 0.5473203063011169
Accuracy: 0.7777777910232544. Recall: 0.5. Precision: 1.0
Fold 5


[I 2024-04-27 13:25:08,100] Trial 0 finished with values: [0.6305555701255798, 0.6000000238418579, 0.6523810029029846, 0.1520436943871626, 0.25495097041130066, 0.19726014137268066] and parameters: {'batch_size': 32, 'lr': 1.774840233277077e-05, 'data_aug': True, 'ini': 'H', 'e_freeze': 4, 'e_all': 48}. 


Best epoc: 2. Best val loss: 0.6978538036346436
Accuracy: 0.375. Recall: 0.75. Precision: 0.4285714328289032
Fold 1
Best epoc: 3. Best val loss: 0.7013362050056458
Accuracy: 0.4444444477558136. Recall: 0.0. Precision: 0.0
Fold 2
Best epoc: 46. Best val loss: 0.6006577610969543
Accuracy: 0.6666666865348816. Recall: 0.5. Precision: 0.6666666865348816
Fold 3
Best epoc: 47. Best val loss: 0.6035686731338501
Accuracy: 0.5555555820465088. Recall: 0.0. Precision: 0.0
Fold 4
Best epoc: 16. Best val loss: 0.6922686696052551
Accuracy: 0.5555555820465088. Recall: 0.25. Precision: 0.5
Fold 5


[I 2024-04-27 13:25:57,978] Trial 1 finished with values: [0.49444445967674255, 0.20000000298023224, 0.2833333373069763, 0.14098420410621115, 0.18708288669586182, 0.2666666723787785] and parameters: {'batch_size': 32, 'lr': 8.616862257643216e-05, 'data_aug': True, 'ini': 'None', 'e_freeze': 6, 'e_all': 46}. 


Best epoc: 48. Best val loss: 0.6225265264511108
Accuracy: 0.25. Recall: 0.25. Precision: 0.25
Fold 1
Best epoc: 22. Best val loss: 0.6375424265861511
Accuracy: 0.8888888955116272. Recall: 1.0. Precision: 0.800000011920929
Fold 2
Best epoc: 39. Best val loss: 0.4241236448287964
Accuracy: 0.4444444477558136. Recall: 0.25. Precision: 0.3333333432674408
Fold 3
Best epoc: 4. Best val loss: 0.7321234345436096
Accuracy: 0.6666666865348816. Recall: 0.25. Precision: 1.0
Fold 4
Best epoc: 41. Best val loss: 0.6975942850112915
Accuracy: 0.6666666865348816. Recall: 0.75. Precision: 0.6000000238418579
Fold 5


... (192 outputs left out)

[I 2024-04-27 14:38:50,266] Trial 98 finished with values: [0.6111111223697663, 0.6000000238418579, 0.7266666293144226, 0.1859244517261509, 0.33911648392677307, 0.26865923404693604] and parameters: {'batch_size': 32, 'lr': 9.780605889544175e-05, 'data_aug': False, 'ini': 'H', 'e_freeze': 7, 'e_all': 39}. 


Best epoc: 16. Best val loss: 0.8389719724655151
Accuracy: 0.5. Recall: 1.0. Precision: 0.5
Fold 1
Best epoc: 0. Best val loss: 0.815302312374115
Accuracy: 0.6666666865348816. Recall: 1.0. Precision: 0.5714285969734192
Fold 2
Best epoc: 2. Best val loss: 0.7069829106330872
Accuracy: 0.5555555820465088. Recall: 0.75. Precision: 0.5
Fold 3
Best epoc: 5. Best val loss: 0.668519914150238
Accuracy: 0.4444444477558136. Recall: 0.0. Precision: 0.0
Fold 4
Best epoc: 19. Best val loss: 0.6953925490379333
Accuracy: 0.6666666865348816. Recall: 1.0. Precision: 0.5714285969734192
Fold 5


[I 2024-04-27 14:39:30,042] Trial 99 finished with values: [0.5416666805744171, 0.699999988079071, 0.41428571939468384, 0.11719458138690825, 0.36742347478866577, 0.21380899846553802] and parameters: {'batch_size': 32, 'lr': 4.6447301334297586e-05, 'data_aug': True, 'ini': 'O', 'e_freeze': 4, 'e_all': 38}. 


Best epoc: 13. Best val loss: 0.6099395751953125
Accuracy: 0.375. Recall: 0.75. Precision: 0.4285714328289032
Best trial:
Value: [0.6361111283302308, 0.6000000238418579, 0.654285728931427, 0.043390274535737024, 0.29999998211860657, 0.17668141424655914]
Params: 
batch_size: 16
lr: 7.039032054564125e-05
data_aug: False
ini: H
e_freeze: 5
e_all: 41
Value: [0.6166666805744171, 0.699999988079071, 0.47317461371421815, 0.10599324422004497, 0.36742347478866577, 0.2557696079960294]
Params: 
batch_size: 16
lr: 1.4477877408260112e-05
data_aug: False
ini: H
e_freeze: 3
e_all: 46
Value: [0.5666666865348816, 0.3499999940395355, 0.4333333373069763, 0.05443310945076494, 0.20000000298023224, 0.22607767020534758]
Params: 
batch_size: 16
lr: 9.780605889544175e-05
data_aug: False
ini: H
e_freeze: 7
e_all: 39
Value: [0.569444453716278, 0.6499999761581421, 0.5323809385299683, 0.1248455861397084, 0.20000000298023224, 0.11438092589378357]
Params: 
batch_size: 16
lr: 3.54589607753455e-05
data_aug: True
ini: O
